## Setup

Main things to note:
1. I (intermediate) was eliminated from the dataset because of its uncertainty to the general dataset -> will be mentioned in the limitations section
2. MIC breakpoints difference were analyzed and these were reported : Total rows : 20,446,868 Successfully relabeled : 18,654,344 (91.2%) NaN (no breakpoint) : 1,792,524 (8.8%)
Comparable rows        : 18,654,344
Labels that change     : 12,468  (0.07%)
Labels unchanged       : 18,641,876
WHich approves the little difference in forward fill and backward fill
3. Sidero index as a isolate id (which may cause duplicates in the dataset), duplicate checks were conducted with no duplicates in both temporal split and overall
4. Time aware CV leak was investigated and resolved below in the notebook
5. class imbalance were handled universally on training the 9 models
6. the hyperparameters are consistent
7. Isotonic Calibration was fitted on validation set [iso.fit(val_preds[name], y_val)] 

In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import GroupKFold
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
os.makedirs('/kaggle/working/', exist_ok=True)
print("Ready")

In [ ]:
ATLAS_PATH    = '/kaggle/input/datasets/uaisamangeldi/atlas-csv/atlas.csv'
SIDERO_PATH   = '/kaggle/input/datasets/uaisamangeldi/ddddddd/sidero_wt.xlsx'
KEYSTONE_PATH = '/kaggle/input/datasets/uaisamangeldi/ddddddd/keystone.xlsx'
COMBINED_OUTPUT = '/kaggle/working/combined_dataset.csv'

def parse_sir_to_binary(sir_str):
    """SIR → binary. Intermediate (I) discarded as clinically ambiguous."""
    if pd.isna(sir_str):
        return np.nan
    s = str(sir_str).strip().upper()
    if s in ['R', 'RESISTANT']:   return 1
    if s in ['S', 'SUSCEPTIBLE']: return 0
    return np.nan

def clean_mic_value(val):
    "Normalize MIC string to float."
    if pd.isna(val): return np.nan
    val = str(val).strip().replace(',', '.')
    for sym in ['<=', '≤', '>=', '≥']:
        val = val.replace(sym, '')
    if '>' in val:
        try: return float(val.replace('>', '').strip()) + 0.0001
        except: return np.nan
    try: return float(val.strip())
    except: return np.nan

def parse_mic_to_binary(mic_val, antibiotic):
    "MIC → binary using simplified CLSI/EUCAST breakpoints."
    mic = clean_mic_value(mic_val)
    if pd.isna(mic): return np.nan
    thresholds = {
        'Meropenem': 4, 'Imipenem': 4, 'Ceftazidime': 8,
        'Ciprofloxacin': 1, 'Colistin': 4, 'Amikacin': 16,
        'Gentamicin': 8, 'Cefepime': 8,
    }
    return 1 if mic > thresholds.get(antibiotic, 8) else 0

def assign_unique_isolate_id(df, data_source):
    "Composite isolate ID: {source}_{original_id}_{year} — prevents cross-database collisions."
    if data_source == 'ATLAS' and 'Isolate Id' in df.columns:
        df['original_id'] = df['Isolate Id'].astype(str).str.strip()
    elif data_source == 'KEYSTONE' and 'Collection Number' in df.columns:
        df['original_id'] = df['Collection Number'].astype(str).str.strip()
    else:
        df['original_id'] = df.index.astype(str)
    df['isolate_id'] = data_source + '_' + df['original_id'] + '_' + df['year'].fillna(0).astype(int).astype(str)
    return df

def process_dataset(path, data_source, chunksize=250000):
    print(f"Processing {data_source}...")
    is_csv = path.endswith('.csv')
    reader = pd.read_csv if is_csv else pd.read_excel
    chunks = reader(path, chunksize=chunksize, low_memory=False) if is_csv else [reader(path)]
    processed = []

    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}: {len(chunk)} rows")
        rename = {
            'Isolate Id': 'original_id_temp', 'Species': 'organism',
            'Country': 'country', 'Year': 'year',
            'Organism Name': 'organism', 'Year Collected': 'year',
            'Organism': 'organism', 'Study Year': 'year',
            'Collection Number': 'original_id_temp',
        }
        chunk = chunk.rename(columns=rename)
        for col in ['country', 'year', 'organism']:
            if col not in chunk.columns:
                chunk[col] = 'Unknown' if col != 'year' else np.nan
        chunk['country']  = chunk['country'].astype(str).str.strip()
        chunk['organism'] = chunk['organism'].astype(str).str.strip()
        chunk['year']     = pd.to_numeric(chunk['year'], errors='coerce').astype('Int16')
        chunk = assign_unique_isolate_id(chunk, data_source)

        sir_cols = [c for c in chunk.columns if c.endswith('_I')]
        mic_cols = [c for c in chunk.columns if c not in
                    sir_cols + ['isolate_id', 'country', 'year', 'organism', 'original_id', 'original_id_temp']]
        value_vars   = sir_cols if sir_cols else mic_cols
        melt_varname = 'antibiotic_raw' if sir_cols else 'antibiotic'
        if not value_vars: continue

        long = pd.melt(chunk, id_vars=['isolate_id', 'country', 'year', 'organism'],
                       value_vars=value_vars, var_name=melt_varname, value_name='test_result')

        if sir_cols:
            long['antibiotic'] = long['antibiotic_raw'].str.replace('_I$', '', regex=True).str.strip()
            long['resistance']  = long['test_result'].apply(parse_sir_to_binary)
        else:
            long['antibiotic'] = long['antibiotic'].str.strip()
            long['resistance']  = long.apply(lambda r: parse_mic_to_binary(r['test_result'], r['antibiotic']), axis=1)

        long = long.dropna(subset=['resistance'])
        long['resistance']   = long['resistance'].astype('int8')
        long['data_source']  = data_source
        processed.append(long)

    result = pd.concat(processed, ignore_index=True)
    print(f"  → {len(result):,} rows")
    return result

atlas    = process_dataset(ATLAS_PATH,    'ATLAS')
sidero   = process_dataset(SIDERO_PATH,   'SIDERO-WT')
keystone = process_dataset(KEYSTONE_PATH, 'KEYSTONE')

combined = pd.concat([atlas, sidero, keystone], ignore_index=True)
combined.to_csv(COMBINED_OUTPUT, index=False)
print(f"\nCombined: {len(combined):,} rows | {combined['isolate_id'].nunique():,} unique isolates")
print(f"Class distribution:\n{combined['resistance'].value_counts(normalize=True).round(4)}")

In [ ]:
full_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
# How many rows are affected by removing backfill?
old = full_df.groupby('country')['gdp_per_capita_usd'].transform(lambda x: x.ffill().bfill())
new = full_df.groupby('country')['gdp_per_capita_usd'].transform(lambda x: x.ffill())
print((old != new).sum(), "rows differ")

In [ ]:
import re

def parse_mic_value(val_str):
    """
    Parses MIC strings like '≤0.5', '>8', '0.25', '4' into floats.
    For inequality operators:
      ≤x  -> x   (at or below breakpoint → likely S)
      <x  -> x*0.5 (conservative)
      ≥x  -> x   (at or above → likely R)  
      >x  -> x*2  (above breakpoint → likely R)
    """
    if pd.isna(val_str):
        return np.nan
    s = str(val_str).strip()
    try:
        return float(s)
    except ValueError:
        pass

    s_clean = s.replace(',', '.')
    m = re.match(r'^([≤<≥>]=?)\s*([\d.]+)$', s_clean)
    if not m:
        return np.nan

    op, num = m.group(1), float(m.group(2))
    if op in ('≤', '<=', '<'):  return num          # treating as exactly at boundary
    if op in ('≥', '>=', '>'):  return num * 2      # pushing above boundary
    return np.nan


def classify_mic_v2(row):
    """
    Reads from test_result column which contains mixed SIR strings + MIC values.
    Uses organism-specific EUCAST breakpoints where available.
    """
    raw = str(row.get('test_result', '')).strip()

    #SIR string path
    raw_upper = raw.upper()
    if raw_upper in ('SUSCEPTIBLE', 'S'):    return 0
    if raw_upper in ('RESISTANT',   'R'):    return 1
    if raw_upper in ('INTERMEDIATE','I'):    return 0  # EUCAST 2019+

    #MIC numeric path
    mic = parse_mic_value(raw)
    if np.isnan(mic):
        return np.nan

    antibiotic = str(row.get('antibiotic', '')).strip()
    organism   = str(row.get('organism',   '')).strip()

    bp = EUCAST_BREAKPOINTS.get((antibiotic, organism))

    if bp is None:
        threshold = FALLBACK_BREAKPOINTS.get(antibiotic)
        if threshold is None:
            return np.nan          
        return int(mic > threshold)

    if mic > bp['R']:   return 1
    if mic <= bp['S']:  return 0
    return 0

In [ ]:

old_labels = full_df['resistance'].copy()

full_df['resistance_new'] = full_df.apply(classify_mic_v2, axis=1)

# How many rows get a new label vs NaN
nan_new    = full_df['resistance_new'].isna().sum()
total      = len(full_df)
labeled    = full_df['resistance_new'].notna().sum()

print(f"Total rows             : {total:,}")
print(f"Successfully relabeled : {labeled:,}  ({labeled/total:.1%})")
print(f"NaN (no breakpoint)    : {nan_new:,}  ({nan_new/total:.1%})")
print()

# Among rows where BOTH old and new labels exist, how many differ?
mask    = old_labels.notna() & full_df['resistance_new'].notna()
changed = (old_labels[mask] != full_df['resistance_new'][mask]).sum()
compare_total = mask.sum()
pct     = changed / compare_total * 100

print(f"Comparable rows        : {compare_total:,}")
print(f"Labels that change     : {changed:,}  ({pct:.2f}%)")
print(f"Labels unchanged       : {compare_total - changed:,}")
print()

# Per-antibiotic breakdown
breakdown = (
    full_df[mask & (old_labels != full_df['resistance_new'])]
    .groupby('antibiotic').size()
    .reset_index(name='n_changed')
    .assign(pct=lambda x: x['n_changed'] / changed * 100)
    .sort_values('n_changed', ascending=False)
    .head(15)
)
print("Top antibiotics driving changes:")
print(breakdown.to_string(index=False))
print()

if pct < 2:
    print("<2% — document as limitation, skip retraining")
elif pct < 5:
    print("2–5% — retrain CatBoost only if time allows")
else:
    print(">5% — retraining needed for valid results")

In [ ]:
# Check what your actual columns are called
print(full_df.columns.tolist())
print()

# Check what the resistance column looks like
print(full_df['resistance'].value_counts(dropna=False).head(10))
print()

# Check what SIR/MIC columns exist
sir_candidates = [c for c in full_df.columns if any(x in c.lower() for x in ['sir', 'suscept', 'mic', 'resist'])]
print("Candidate columns:", sir_candidates)
print()

# Sample a few rows to see the raw data
print(full_df[sir_candidates + ['antibiotic', 'organism']].head(10))

In [ ]:
# GLOBAL DATASET
import os
import gc
import pandas as pd
import numpy as np

# ── Original Paths ───────────────────────────────────────────────────────────
POP_DENSITY_PATH = '/kaggle/input/datasets/uaisamangeldi/macrob/API_EN.POP.DNST_DS2_en_csv_v2_275/API_EN.POP.DNST_DS2_en_csv_v2_275.csv'
GDP_PCAP_PATH    = '/kaggle/input/datasets/uaisamangeldi/macrob/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_31/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_31.csv'
HEALTH_GDP_PATH  = '/kaggle/input/datasets/uaisamangeldi/macrob/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_938/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_938.csv'
CONSUMPTION_PATH = '/kaggle/input/datasets/uaisamangeldi/consumption/antibiotic-consumption-rate.csv'
GLOBAL_DID_CLASS_PATH = '/kaggle/input/datasets/uaisamangeldi/globalddd/antibiotic_consumption_clean.csv'

# ── NEW FEATURE PATHS ───────────────────────────────────────────────────────
SANITATION_PATH    = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874.csv'
WATER_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137.csv'
HOSPITAL_BEDS_PATH = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206.csv'
TEMPERATURE_PATH   = '/kaggle/input/datasets/uaisamangeldi/highfeature/average-monthly-surface-temperature/average-monthly-surface-temperature.csv'
AWARE_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/data.csv'

# ── Helper: downcast numerics ────────────────────────────────────────────────
def downcast(df):
    for col in df.select_dtypes(include='float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

# ── Helper: WB long format ───────────────────────────────────────────────────
def wb_long_format(path, value_col_name):
    if not os.path.exists(path):
        print(f" [MISSING] {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, skiprows=4, low_memory=False)
    df.columns = df.columns.str.strip()
    if 'Country Name' not in df.columns:
        print(f" [ERROR] 'Country Name' missing in {path}")
        return pd.DataFrame()
    year_cols = [c for c in df.columns if c.strip().isdigit() and int(c.strip()) >= 2000]
    df_long = pd.melt(df, id_vars=['Country Name'], value_vars=year_cols,
                      var_name='year', value_name=value_col_name)
    del df; gc.collect()
    df_long['year'] = df_long['year'].astype(int)
    df_long = df_long.rename(columns={'Country Name': 'country'})
    df_long[value_col_name] = pd.to_numeric(df_long[value_col_name], errors='coerce')
    result = df_long[['country', 'year', value_col_name]].dropna(subset=[value_col_name])
    del df_long; gc.collect()
    result = downcast(result)
    result['country'] = result['country'].astype('category')
    print(f" {value_col_name}: {len(result):,} rows, {result['country'].nunique()} countries")
    return result

# ── Country Mapping ─────────────────────────────────────────────────────────
COUNTRY_MAP = {
    'United States': 'USA', 'United States of America': 'USA',
    'Russian Federation': 'Russia', 'Korea, Rep.': 'South Korea',
}

# ── Load full_df with categorical strings ────────────────────────────────────
full_df = pd.read_csv("/kaggle/working/combined_dataset.csv")
full_df['country'] = full_df['country'].astype('category')

# ── Core World Bank Indicators ──────────────────────────────────────────────
print("Loading World Bank indicators...")
df_gdp  = wb_long_format(GDP_PCAP_PATH,    'gdp_per_capita_usd')
df_pop  = wb_long_format(POP_DENSITY_PATH, 'population_density_per_sq_km')
df_health = wb_long_format(HEALTH_GDP_PATH, 'health_expenditure_pct_gdp')

macro_df = df_gdp.copy()
del df_gdp; gc.collect()

for df_temp in [df_pop, df_health]:
    macro_df = macro_df.merge(df_temp, on=['country', 'year'], how='outer')
    del df_temp; gc.collect()

del df_pop, df_health; gc.collect()

# ── New World Bank Indicators ───────────────────────────────────────────────
print("\nLoading new World Bank indicators...")
for path, name in [
    (SANITATION_PATH,    'sanitation_access_pct'),
    (WATER_PATH,         'water_access_pct'),
    (HOSPITAL_BEDS_PATH, 'hospital_beds_per_1000'),
]:
    df_temp = wb_long_format(path, name)
    if not df_temp.empty:
        macro_df = macro_df.merge(df_temp, on=['country', 'year'], how='left')
    del df_temp; gc.collect()

# ── Temperature (Parse year from 'Day' column) ──────────────────────────────
print("\nLoading & aggregating temperature data...")
try:
    df_temp = pd.read_csv(TEMPERATURE_PATH, usecols=['Entity', 'Day', 'Monthly average'])
    print(f"Raw temperature shape: {df_temp.shape}")

    df_temp['year'] = pd.to_datetime(df_temp['Day'], errors='coerce').dt.year
    df_temp = df_temp.drop(columns=['Day'])  # free Day column immediately

    df_temp_yearly = df_temp.groupby(['Entity', 'year'], as_index=False)['Monthly average'].mean()
    del df_temp; gc.collect()

    df_temp_yearly = df_temp_yearly.rename(columns={
        'Entity': 'country',
        'Monthly average': 'mean_temp_celsius'
    })
    df_temp_yearly['country'] = df_temp_yearly['country'].replace(COUNTRY_MAP)
    df_temp_yearly = df_temp_yearly[['country', 'year', 'mean_temp_celsius']].dropna()
    df_temp_yearly['year'] = df_temp_yearly['year'].astype(int)
    df_temp_yearly = downcast(df_temp_yearly)

    macro_df = macro_df.merge(df_temp_yearly, on=['country', 'year'], how='left')
    del df_temp_yearly; gc.collect()
    print(f" mean_temp_celsius: merged OK")
except Exception as e:
    print(f" Temperature FAILED: {e}")
    macro_df['mean_temp_celsius'] = np.nan

# ── AWaRe Access % ──────────────────────────────────────────────────────────
print("\nLoading AWaRe Access % data...")
try:
    df_aware = pd.read_csv(AWARE_PATH, usecols=['Location', 'Period', 'FactValueNumeric'])
    df_aware = df_aware.rename(columns={
        'Location':         'country',
        'Period':           'year',
        'FactValueNumeric': 'aware_access_pct',
    })
    df_aware['country'] = df_aware['country'].replace(COUNTRY_MAP)
    df_aware = df_aware[['country', 'year', 'aware_access_pct']].dropna()
    df_aware['year'] = df_aware['year'].astype(int)
    df_aware = downcast(df_aware)

    macro_df = macro_df.merge(df_aware, on=['country', 'year'], how='left')
    del df_aware; gc.collect()
    print(f" aware_access_pct: merged OK")
except Exception as e:
    print(f" AWaRe FAILED: {e}")
    macro_df['aware_access_pct'] = np.nan

# ── Downcast macro_df before merging into full_df ───────────────────────────
macro_df = downcast(macro_df)
gc.collect()

# ── Cleanup & Antibiotic Class Mapping ──────────────────────────────────────
full_df['country'] = full_df['country'].astype(str).replace(COUNTRY_MAP).astype('category')
full_df = full_df[~full_df['antibiotic'].str.strip().isin(['Age', 'Date Collected', 'age', 'date collected'])]

ANTIBIOTIC_TO_CLASS_MAP = {
    'Meropenem': 'Carbapenems',           'Imipenem': 'Carbapenems',
    'Doripenem': 'Carbapenems',           'Ertapenem': 'Carbapenems',
    'Ciprofloxacin': 'Fluoroquinolones',  'Levofloxacin': 'Fluoroquinolones',
    'Moxifloxacin': 'Fluoroquinolones',
    'Ceftazidime': 'Cephalosporins',      'Cefepime': 'Cephalosporins',
    'Ceftriaxone': 'Cephalosporins',
    'Amikacin': 'Aminoglycosides',        'Gentamicin': 'Aminoglycosides',
    'Ampicillin': 'Penicillins',          'Amoxycillin clavulanate': 'Penicillins',
    'Piperacillin': 'Penicillins',
    'Azithromycin': 'Macrolides',         'Erythromycin': 'Macrolides',
    'Vancomycin': 'Glycopeptides',        'Linezolid': 'Oxazolidinones',
    'Colistin': 'Polymyxins',             'Tigecycline': 'Tetracyclines',
    'Trimethoprim sulfa': 'Sulfonamides',
}

full_df['antibiotic_class'] = full_df['antibiotic'].map(ANTIBIOTIC_TO_CLASS_MAP).astype('category')
full_df['antibiotic']       = full_df['antibiotic'].astype('category')

# ── Drop Raw Columns (Keep antibiotic_class) ────────────────────────────────
drop_cols = [
    'gdp_per_capita_usd', 'population_density_per_sq_km', 'health_expenditure_pct_gdp',
    'consumption_ddd_per_1000_day', 'consumption_ddd_total', 'consumption_did_per_class',
    'log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class',
    'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000',
    'mean_temp_celsius', 'aware_access_pct',
]
full_df = full_df.drop(columns=[c for c in drop_cols if c in full_df.columns])

full_df = full_df.merge(macro_df, on=['country', 'year'], how='left')
del macro_df; gc.collect()

# ── Total Consumption ────────────────────────────────────────────────────────
print("\nLoading total consumption...")
try:
    df_cons = pd.read_csv(CONSUMPTION_PATH, usecols=[
        'Entity', 'Year',
        'Defined daily doses of antibiotics and antituberculosis drugs used per 1,000 inhabitants per day'
    ])
    df_cons = df_cons.rename(columns={
        'Entity': 'country',
        'Year':   'year',
        'Defined daily doses of antibiotics and antituberculosis drugs used per 1,000 inhabitants per day': 'consumption_ddd_total'
    })
    df_cons['country'] = df_cons['country'].replace({'Russian Federation': 'Russia', 'United States': 'USA'})
    df_cons = downcast(df_cons)

    full_df = full_df.merge(df_cons, on=['country', 'year'], how='left')
    del df_cons; gc.collect()
    print(" → Total consumption OK")
except Exception as e:
    print(f" → Total consumption FAILED: {e}")
    full_df['consumption_ddd_total'] = np.nan

# ── Per-class DDD ────────────────────────────────────────────────────────────
print("\nLoading per-class DDD...")
did_raw = pd.read_csv(GLOBAL_DID_CLASS_PATH,
                      usecols=['country', 'year', 'antibiotic_class', 'consumption_DID'])
DID_COUNTRY_MAP = {
    'Us': 'USA', 'Uk': 'United Kingdom', 'Uae': 'United Arab Emirates',
    'Korea': 'South Korea', 'Russian Federation': 'Russia'
}
did_raw['country'] = did_raw['country'].replace(DID_COUNTRY_MAP)
did_raw = did_raw.rename(columns={'consumption_DID': 'consumption_did_per_class'})
did_raw = downcast(did_raw)

full_df = full_df.merge(did_raw, on=['country', 'year', 'antibiotic_class'], how='left')
del did_raw; gc.collect()

# ── Imputation ──────────────────────────────────────────────────────────────
fill_cols = [
    'gdp_per_capita_usd', 'population_density_per_sq_km', 'health_expenditure_pct_gdp',
    'consumption_ddd_total', 'consumption_did_per_class',
    'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000',
    'mean_temp_celsius', 'aware_access_pct'
]

# Convert categoricals back to string for groupby compatibility
for col in ['country', 'antibiotic', 'antibiotic_class']:
    if col in full_df.columns and full_df[col].dtype.name == 'category':
        full_df[col] = full_df[col].astype(str)

for col in fill_cols:
    if col in full_df.columns:
        full_df[col] = full_df.groupby('country')[col].transform(lambda x: x.ffill().bfill())
        full_df[col] = full_df[col].fillna(full_df[col].mean())

# ── Log Transforms ──────────────────────────────────────────────────────────
full_df['log_gdp_per_capita']            = np.log1p(full_df['gdp_per_capita_usd'].clip(lower=0))
full_df['log_consumption_ddd']           = np.log1p(full_df['consumption_ddd_total'].clip(lower=0))
full_df['log_consumption_did_per_class'] = np.log1p(full_df['consumption_did_per_class'].clip(lower=0))

full_df = downcast(full_df)
gc.collect()

# ── Save ────────────────────────────────────────────────────────────────────
os.makedirs('/kaggle/working', exist_ok=True)
full_df.to_csv('/kaggle/working/full_amr_with_macro.csv', index=False)

print(f"\n✅ Successfully saved full_amr_with_macro.csv | Shape: {full_df.shape}")
print("\nCoverage (non-null %):")
print((full_df[fill_cols + ['log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class']]
       .notna().mean() * 100).round(1))

In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------------------------------
# 1. Load raw data
# --------------------------------------------------------------------------
RAW_PATH = '/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv'
raw = pd.read_csv(RAW_PATH)
total_raw = len(raw)
print(f"Raw CSV rows: {total_raw:,}")

has_label = raw['resistance'].notna()
n_missing_label = total_raw - has_label.sum()
labeled = raw[has_label].copy()


has_year = labeled['year'].notna()
n_missing_year = has_label.sum() - has_year.sum()
analysis_set = labeled[has_year].copy()


TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020

train = analysis_set[analysis_set['year'] <= TRAIN_CUTOFF]
val   = analysis_set[(analysis_set['year'] > TRAIN_CUTOFF) & (analysis_set['year'] <= VAL_CUTOFF)]
test  = analysis_set[analysis_set['year'] > VAL_CUTOFF]

attrition = pd.DataFrame({
    'Step': [
        '1. Raw dataset',
        '2. Remove rows with missing resistance label',
        '3. Remove rows with missing year',
        '4. Train set (≤ 2018)',
        '5. Validation set (2019–2020)',
        '6. Test set (> 2020)'
    ],
    'Rows': [
        total_raw,
        n_missing_label,
        n_missing_year,
        len(train),
        len(val),
        len(test)
    ],
    'Remaining': [
        total_raw,
        total_raw - n_missing_label,
        total_raw - n_missing_label - n_missing_year,
        len(train),
        len(val),
        len(test)
    ]
})

# Add class balance (resistance prevalence) for the final three splits
def class_balance(df):
    n_total = len(df)
    n_pos = df['resistance'].sum()
    n_neg = n_total - n_pos
    return f"Resistant: {n_pos:,} ({n_pos/n_total:.1%}) | Susceptible: {n_neg:,} ({n_neg/n_total:.1%})"

attrition['Class Balance (Resistant / Susceptible)'] = [
    '',   
    '',  
    '',   
    class_balance(train),
    class_balance(val),
    class_balance(test)
]

# Make the table look like a proper publication table
attrition['Remaining'] = attrition['Remaining'].apply(lambda x: f"{x:,}")
attrition['Rows'] = attrition['Rows'].apply(lambda x: f"{x:,}")

print("\nTable 1 — Data Attrition Summary\n")
print(attrition.to_string(index=False))

attrition.to_csv('/kaggle/working/table1_attrition.csv', index=False)
print("\n Saved: /kaggle/working/table1_attrition.csv")

In [ ]:
import pandas as pd

# --------------------------------------------------------------------------
# 1. Load the Kazakhstan predictions
# --------------------------------------------------------------------------
path = '/kaggle/input/datasets/uaisamangeldi/kzzzzz/kz_loco_fixed (1).csv'
df = pd.read_csv(path)

# Show available columns so you can pick the right probability column
print("Columns:", df.columns.tolist())

# --------------------------------------------------------------------------
# 2. Choose the column that represents predicted resistance risk
#    (adjust the name if yours is different – likely 'prob_loco_ensemble')
# --------------------------------------------------------------------------
PROB_COL = 'prob_loco_ensemble'     # change to your actual column name
if PROB_COL not in df.columns:
    # fallback: try to use any column that starts with 'prob_'
    prob_candidates = [c for c in df.columns if c.startswith('prob_')]
    if prob_candidates:
        PROB_COL = prob_candidates[0]   # take the first one
        print(f"⚠️  'prob_loco_ensemble' not found. Using '{PROB_COL}' instead.")
    else:
        raise KeyError("No probability column found! Check the CSV columns above.")

# --------------------------------------------------------------------------
# 3. Create a combined "combo" label
# --------------------------------------------------------------------------
df['combo'] = df['organism'] + ' + ' + df['antibiotic']

# --------------------------------------------------------------------------
# 4. For each year, find the top‑5 combos by predicted risk
# --------------------------------------------------------------------------
top5_per_year = []
for year in sorted(df['year'].unique()):
    subset = df[df['year'] == year].copy()
    # Some combos might appear multiple times (e.g., different macro rows) – take the mean
    avg_risk = subset.groupby('combo')[PROB_COL].mean().reset_index()
    top5 = avg_risk.nlargest(5, PROB_COL)
    top5.insert(0, 'year', year)
    top5_per_year.append(top5)

top5_table = pd.concat(top5_per_year, ignore_index=True)

# Clean up formatting: rename and round
top5_table = top5_table.rename(columns={PROB_COL: 'predicted_risk'})
top5_table['predicted_risk'] = top5_table['predicted_risk'].round(4)

# --------------------------------------------------------------------------
# 5. Print the final table
# --------------------------------------------------------------------------
print("\nTable 3 — Top‑5 highest‑risk antibiotic‑organism combos per year (Kazakhstan)\n")
print(top5_table.to_string(index=False))

# Save to CSV
top5_table.to_csv('/kaggle/working/table3_top5_risk_combos.csv', index=False)
print("\n Saved: /kaggle/working/table3_top5_risk_combos.csv")

In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------------------------------
# 1. Load the Kazakhstan predictions
# --------------------------------------------------------------------------
path = '/kaggle/input/datasets/uaisamangeldi/kzzzzz/kz_loco_fixed (1).csv'
df = pd.read_csv(path)

# --------------------------------------------------------------------------
# 2. Choose the probability column (predicted resistance risk)
# --------------------------------------------------------------------------
PROB_COL = 'prob_loco_ensemble'
if PROB_COL not in df.columns:
    prob_candidates = [c for c in df.columns if c.startswith('prob_')]
    if prob_candidates:
        PROB_COL = prob_candidates[0]
        print(f"⚠️  'prob_loco_ensemble' not found. Using '{PROB_COL}'")
    else:
        raise KeyError("No probability column found! Check columns:", df.columns.tolist())

# --------------------------------------------------------------------------
# 3. Collapse by organism × year (risk is identical for antibiotics of same class)
# --------------------------------------------------------------------------
grouped = df.groupby(['year', 'organism']).agg(
    max_risk=(PROB_COL, 'max'),                     # = any value, they're all identical
    antibiotics=('antibiotic', lambda x: ', '.join(sorted(x.unique())))
).reset_index()

# Keep only the top 5 organisms by risk per year
top5_per_year = (grouped
                 .sort_values(['year', 'max_risk'], ascending=[True, False])
                 .groupby('year')
                 .head(5)
                 .reset_index(drop=True))

# Round risk for clean display
top5_per_year['max_risk'] = top5_per_year['max_risk'].round(4)

# --------------------------------------------------------------------------
# 4. Rename for publication
# --------------------------------------------------------------------------
top5_per_year = top5_per_year.rename(columns={
    'year': 'Year',
    'organism': 'Organism',
    'max_risk': 'Highest predicted risk',
    'antibiotics': 'Notable antibiotics (same risk)'
})

# --------------------------------------------------------------------------
# 5. Print and save
# --------------------------------------------------------------------------
print("\nTable 3 — Top‑5 highest‑risk organisms per year for Kazakhstan\n")
print(top5_per_year.to_string(index=False))
top5_per_year.to_csv('/kaggle/working/table3_top5_orgs_kz.csv', index=False)
print("\n Saved: /kaggle/working/table3_top5_orgs_kz.csv")

## Features


In [ ]:
CAT_FEATURES = [
    'organism', 
    'antibiotic', 
    'antibiotic_class'         
]

NUM_FEATURES = [
    'year',
    'log_gdp_per_capita',
    'health_expenditure_pct_gdp',
    'population_density_per_sq_km',
    'log_consumption_ddd',
    'log_consumption_did_per_class',
    'sanitation_access_pct',
    'water_access_pct',
    'hospital_beds_per_1000',
    'mean_temp_celsius',
    'aware_access_pct'
]

FEATURES = CAT_FEATURES + NUM_FEATURES

print("Categorical features :", CAT_FEATURES)
print("Numerical features   :", NUM_FEATURES)
print("Total features       :", len(FEATURES))
print("\n Feature list ready for training.")

In [ ]:
import pandas as pd

# Load the file with LOCO predictions (generated from previous LOCO code)
kz_pred = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/kzzzzz/kz_loco_fixed (1).csv')

print(f"Total prediction rows: {len(kz_pred):,}\n")

# Select relevant columns and sort by ensemble probability (most reliable)
top_risks = kz_pred[[
    'year', 
    'antibiotic', 
    'antibiotic_class',
    'organism', 
    'prob_loco_ensemble_raw'
]].copy()

# Sort by highest predicted resistance probability
top_risks = top_risks.sort_values(by='prob_loco_ensemble_raw', ascending=False)

# Show Top 10
print("TOP 10 HIGHEST RISK ANTIBIOTIC-ORGANISM COMBINATIONS IN KAZAKHSTAN\n")
print(top_risks.head(10).round(4).to_string(index=False))

# Save full ranked list
top_risks.to_csv('/kaggle/working/kazakhstan_top_risk_combinations.csv', index=False)

print("\nFull ranked list saved as: kazakhstan_top_risk_combinations.csv")

In [ ]:
# Top 5 highest risk per year
print("TOP 5 HIGHEST RISK COMBINATIONS PER YEAR:\n")
for year in sorted(top_risks['year'].unique()):
    yearly = top_risks[top_risks['year'] == year].head(5)
    print(f"Year {year}:")
    print(yearly[['antibiotic', 'organism', 'prob_loco_ensemble_raw']].round(4).to_string(index=False))
    print("-" * 60)

In [ ]:
!pip install catboost
!pip install interpret
!pip install ngboost
!pip install autogluon.tabular 

# Catboost

In [ ]:
import gc
import pickle
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    brier_score_loss, roc_auc_score, average_precision_score
)


#memory cleanup

def cleanup(*args):
    for obj in args:
        del obj
    gc.collect()
#data loading
TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020
LABEL_ENCODERS_PATH = '/kaggle/input/datasets/uaisamangeldi/importmodels/all_mlp_files/label_encoders.pkl'

source_df     = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)
cleanup(source_df)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()
cleanup(source_sorted)

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})")

X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values.astype(np.int8)
y_val   = val_df['resistance'].values.astype(np.int8)
y_test  = test_df['resistance'].values.astype(np.int8)

cleanup(train_df, val_df, test_df)

#catboost prep

for col in CAT_FEATURES:
    X_train[col] = X_train[col].fillna('unknown').astype(str)
    X_val[col]   = X_val[col].fillna('unknown').astype(str)
    X_test[col]  = X_test[col].fillna('unknown').astype(str)

for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype(np.float32)
    X_val[col]   = X_val[col].astype(np.float32)
    X_test[col]  = X_test[col].astype(np.float32)

#catboost train

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight : {pos_weight:.2f}")
print("\n Training CatBoost")

cb_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3,
    scale_pos_weight=pos_weight,
    cat_features=CAT_FEATURES,
    eval_metric='AUC',
    early_stopping_rounds=100,
    random_seed=42,
    verbose=200,
    task_type='GPU',
)

cb_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

#evaluate

val_probs  = cb_model.predict_proba(X_val)[:, 1].astype(np.float32)
test_probs = cb_model.predict_proba(X_test)[:, 1].astype(np.float32)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_bin = (y_prob >= threshold).astype(int)
    prevalence = y_true.mean()
    bs     = brier_score_loss(y_true, y_prob)
    bs_ref = prevalence * (1 - prevalence)
    bss    = 1.0 - bs / bs_ref if bs_ref > 0 else np.nan
    return {
        'auc'       : roc_auc_score(y_true, y_prob),
        'auprc'     : average_precision_score(y_true, y_prob),
        'f1'        : f1_score(y_true, y_bin),
        'precision' : precision_score(y_true, y_bin),
        'recall'    : recall_score(y_true, y_bin),
        'brier'     : bs,
        'bss'       : bss,
    }

metrics = compute_metrics(y_test, test_probs)
print(f"  AUC={metrics['auc']:.4f} | "
      f"AUPRC={metrics['auprc']:.4f} | "
      f"Brier={metrics['brier']:.4f} | "
      f"F1={metrics['f1']:.4f}")

#save

cb_model.save_model('/kaggle/working/catboost_model.cbm')
np.save('/kaggle/working/catboost_val_preds.npy',  val_probs)
np.save('/kaggle/working/catboost_test_preds.npy', test_probs)
np.save('/kaggle/working/y_test.npy', y_test)

#save X_test.npy

with open(LABEL_ENCODERS_PATH, 'rb') as f:
    label_encoders = pickle.load(f)

X_test_encoded = X_test.copy()
for col in CAT_FEATURES:
    le = label_encoders[col]
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    X_test_encoded[col] = X_test_encoded[col].astype(str).map(
        lambda s, d=mapping: d.get(s, -1)   # -1 for unseen categories
    )

np.save('/kaggle/working/X_test.npy', X_test_encoded.values.astype(np.float32))
print("X_test.npy saved with consistent label encoding")

cleanup(X_train, X_val, X_test, X_test_encoded, y_train, y_val, y_test, cb_model)
print("CatBoost saved.")

# Model Download 

In [ ]:
import base64, os
from IPython.display import HTML

zip_path = '/kaggle/working/all_cbm_files.zip'

with open(zip_path, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f'''
<script>
var link = document.createElement('a');
link.href = 'data:application/zip;base64,{b64}';
link.download = '{os.path.basename(zip_path)}';
document.body.appendChild(link);
link.click();
document.body.removeChild(link);
</script>
'''))
print("Download should start immediately")

# MLP

In [ ]:
import gc
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import pickle

#memory cleanup
def cleanup(*args):
    for obj in args:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
#data loading

TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020

source_df     = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)
cleanup(source_df)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()
cleanup(source_sorted)

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})")

#label encoding

label_encoders = {}

X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values.astype(np.int8)
y_val   = val_df['resistance'].values.astype(np.int8)
y_test  = test_df['resistance'].values.astype(np.int8)

cleanup(train_df, val_df, test_df)

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('unknown').astype(str))
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_val[col]  = X_val[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    X_test[col] = X_test[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    label_encoders[col] = le

for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype(np.float32)
    X_val[col]   = X_val[col].astype(np.float32)
    X_test[col]  = X_test[col].astype(np.float32)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight : {pos_weight:.2f}")

with open('/kaggle/working/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

# MLP
print("\n MLP")

scaler_mlp         = StandardScaler()
X_train_scaled     = scaler_mlp.fit_transform(X_train.values.astype(np.float32))
X_val_scaled       = scaler_mlp.transform(X_val.values.astype(np.float32))
X_test_scaled      = scaler_mlp.transform(X_test.values.astype(np.float32))

#save scaler and y arrays
with open('/kaggle/working/scaler_mlp.pkl', 'wb') as f:
    pickle.dump(scaler_mlp, f)
np.save('/kaggle/working/y_train.npy', y_train)
np.save('/kaggle/working/y_val.npy',   y_val)
np.save('/kaggle/working/y_test.npy',  y_test)

#save encoded X arrays 
np.save('/kaggle/working/X_train.npy', X_train.values.astype(np.float32))
np.save('/kaggle/working/X_val.npy',   X_val.values.astype(np.float32))
np.save('/kaggle/working/X_test.npy',  X_test.values.astype(np.float32))

cleanup(X_train, X_val, X_test)

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_mem = device.type == 'cuda'
print(f"  MLP device → {device}")

train_tensor = TensorDataset(
    torch.tensor(X_train_scaled),
    torch.tensor(y_train, dtype=torch.float32))
val_tensor   = TensorDataset(
    torch.tensor(X_val_scaled),
    torch.tensor(y_val,   dtype=torch.float32))
test_tensor  = TensorDataset(
    torch.tensor(X_test_scaled),
    torch.tensor(y_test,  dtype=torch.float32))

input_dim = X_train_scaled.shape[1]
cleanup(X_train_scaled, X_val_scaled, X_test_scaled)

train_loader = DataLoader(train_tensor, batch_size=4096, shuffle=True,  pin_memory=pin_mem)
val_loader   = DataLoader(val_tensor,   batch_size=4096, shuffle=False, pin_memory=pin_mem)
test_loader  = DataLoader(test_tensor,  batch_size=4096, shuffle=False, pin_memory=pin_mem)

class AMR_MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

mlp               = AMR_MLP(input_dim).to(device)
pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(device)
criterion         = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer         = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_auc     = 0.0
best_weights     = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
PATIENCE         = 10
patience_counter = 0

for epoch in range(100):
    mlp.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(mlp(xb), yb).backward()
        optimizer.step()

    mlp.eval()
    val_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            val_preds.append(torch.sigmoid(mlp(xb.to(device))).cpu().numpy())
    val_auc = roc_auc_score(y_val, np.concatenate(val_preds))

    if val_auc > best_val_auc:
        best_val_auc     = val_auc
        best_weights     = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d} — Val AUC: {val_auc:.4f}")

mlp.load_state_dict(best_weights)
mlp.eval()
torch.save(best_weights, '/kaggle/working/mlp_weights.pt')

#generate and save MLP predictions

mlp_test_preds, mlp_val_preds = [], []
with torch.no_grad():
    for xb, _ in test_loader:
        mlp_test_preds.append(torch.sigmoid(mlp(xb.to(device))).cpu().numpy())
    for xb, _ in val_loader:
        mlp_val_preds.append(torch.sigmoid(mlp(xb.to(device))).cpu().numpy())

mlp_test_preds = np.concatenate(mlp_test_preds)
mlp_val_preds  = np.concatenate(mlp_val_preds)

assert not np.isnan(mlp_test_preds).any(), "NaNs in mlp_test_preds"
assert not np.isnan(mlp_val_preds).any(),  "NaNs in mlp_val_preds"

print(f"  MLP Val  AUC : {roc_auc_score(y_val, mlp_val_preds):.4f}")
print(f"  MLP Test AUC : {roc_auc_score(y_test, mlp_test_preds):.4f}")

np.save('/kaggle/working/mlp_val_preds.npy',  mlp_val_preds)
np.save('/kaggle/working/mlp_test_preds.npy', mlp_test_preds)

cleanup(train_tensor, val_tensor, test_tensor, train_loader, val_loader, test_loader)
print("MLP done.")

#download files 

for f in ['mlp_weights.pt', 'mlp_val_preds.npy', 'mlp_test_preds.npy',
          'scaler_mlp.pkl', 'label_encoders.pkl',
          'X_train.npy', 'X_val.npy', 'X_test.npy',
          'y_train.npy', 'y_val.npy', 'y_test.npy']:
    path = f'/kaggle/working/{f}'
    exists = 'Done' if os.path.exists(path) else 'MISSING'
    print(f"  {exists}  {f}")

# LightGBM

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

#memory cleanup

def cleanup(*args):
    for obj in args:
        del obj
    gc.collect()

#data loading 

TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020

source_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)
cleanup(source_df)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()
cleanup(source_sorted)

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})")


X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values.astype(np.int8)
y_val   = val_df['resistance'].values.astype(np.int8)
y_test  = test_df['resistance'].values.astype(np.int8)

# free full dataframes

cleanup(train_df, val_df, test_df)

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('unknown').astype(str))
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_val[col]  = X_val[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    X_test[col] = X_test[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))

# num -> float32
for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype(np.float32)
    X_val[col]   = X_val[col].astype(np.float32)
    X_test[col]  = X_test[col].astype(np.float32)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight : {pos_weight:.2f}")

print("\n LightGBM")

lgb_train = lgb.Dataset(X_train.values, label=y_train, free_raw_data=True)
lgb_val   = lgb.Dataset(X_val.values,   label=y_val,   reference=lgb_train, free_raw_data=True)

lgb_params = {
    'objective':         'binary',
    'metric':            ['auc', 'aucpr'],
    'learning_rate':     0.025,
    'max_depth':         6,
    'num_leaves':        48,
    'subsample':         0.75,
    'colsample_bytree':  0.75,
    'min_data_in_leaf':  60,
    'min_split_gain':    0.02,
    'lambda_l1':         1.0,
    'lambda_l2':         3.0,
    'scale_pos_weight':  pos_weight,
    'device':            'cpu',
    'random_state':      42,
    'n_jobs':            -1,
    'verbose':           -1,
}

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=5000,                       
    valid_sets=[lgb_val],
    callbacks=[
        lgb.early_stopping(stopping_rounds=150, verbose=True),   
        lgb.log_evaluation(period=100),
    ],
)

# pred on validation & test sets
lgb_val_preds  = lgb_model.predict(X_val.values)
lgb_test_preds = lgb_model.predict(X_test.values)

#check
assert not np.isnan(lgb_val_preds).any(),  "NaNs in lgb_val_preds"
assert not np.isnan(lgb_test_preds).any(), "NaNs in lgb_test_preds"

print(f"  LightGBM Val  AUC : {roc_auc_score(y_val,  lgb_val_preds):.4f}")
print(f"  LightGBM Test AUC : {roc_auc_score(y_test, lgb_test_preds):.4f}")

# save preds & model
np.save('/kaggle/working/lgb_val_preds.npy',  lgb_val_preds)
np.save('/kaggle/working/lgb_test_preds.npy', lgb_test_preds)
lgb_model.save_model('/kaggle/working/lgb_model.txt')

cleanup(lgb_train, lgb_val, lgb_model, X_train, X_val, X_test, y_train, y_val, y_test)
gc.collect()
print("LightGBM saved.\n")

# EBM

In [ ]:
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
import pandas as pd
import pickle
from ngboost import NGBClassifier
from ngboost.distns import Bernoulli
from interpret.glassbox import ExplainableBoostingClassifier
from autogluon.tabular import TabularPredictor

def cleanup(*args):
    for obj in args:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020

source_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)
cleanup(source_df)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()
cleanup(source_sorted)

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})")

label_encoders = {}

X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values.astype(np.int8)
y_val   = val_df['resistance'].values.astype(np.int8)
y_test  = test_df['resistance'].values.astype(np.int8)

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('unknown').astype(str))
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_val[col]  = X_val[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    X_test[col] = X_test[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    label_encoders[col] = le

for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype(np.float32)
    X_val[col]   = X_val[col].astype(np.float32)
    X_test[col]  = X_test[col].astype(np.float32)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight : {pos_weight:.2f}")


print("\n Training EBM")

sample_weights_ebm = np.where(y_train == 1, pos_weight, 1.0)

ebm_model = ExplainableBoostingClassifier(
    random_state=42,
    interactions=5,
    learning_rate=0.01,
    max_bins=256
)


ebm_model.fit(X_train.values, y_train, sample_weight=sample_weights_ebm)


with open('/kaggle/working/ebm_model.pkl', 'wb') as f:
    pickle.dump(ebm_model, f)

cleanup(sample_weights_ebm)
print("  EBM trained and saved.")

with open('/kaggle/working/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

print("Saved to : ")
print("ebm_model.pkl")

# LG, RF, XGB, AutoGluon

In [ ]:
import gc
import numpy as np
import pandas as pd
import pickle
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder

#memory cleanup

def cleanup(*args):
    for obj in args:
        del obj
    gc.collect()

#data loading
TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020

source_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)
cleanup(source_df)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()
cleanup(source_sorted)

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})")


label_encoders = {}

X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values.astype(np.int8)
y_val   = val_df['resistance'].values.astype(np.int8)
y_test  = test_df['resistance'].values.astype(np.int8)

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('unknown').astype(str))
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_val[col]  = X_val[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    X_test[col] = X_test[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    label_encoders[col] = le

for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype(np.float32)
    X_val[col]   = X_val[col].astype(np.float32)
    X_test[col]  = X_test[col].astype(np.float32)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight : {pos_weight:.2f}")

# LG
print("\nTraining Logistic Regression...")
scaler_lr = StandardScaler()
X_train_lr = scaler_lr.fit_transform(X_train)
X_val_lr   = scaler_lr.transform(X_val)

lr_model = LogisticRegression(
    class_weight='balanced', max_iter=1000,
    solver='saga', random_state=42, n_jobs=-1
)
lr_model.fit(X_train_lr, y_train)

with open('/kaggle/working/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
with open('/kaggle/working/scaler_lr.pkl', 'wb') as f:
    pickle.dump(scaler_lr, f)

cleanup(X_train_lr, X_val_lr)
print("Logistic Regression saved.")

# RF
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train.values, y_train)

with open('/kaggle/working/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
cleanup(rf_model)
print("  Random Forest saved.")

#XGBoost

print("\nTraining XGBoost...")
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'aucpr'],
    'max_depth': 7,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'scale_pos_weight': pos_weight,
    'tree_method': 'hist',
    'device': 'cuda' if xgb.rabit.get_world_size() == 0 else 'cpu',
    'random_state': 42,
    'n_jobs': -1,
}
dtrain = xgb.DMatrix(X_train.values, label=y_train, feature_names=FEATURES)
dval   = xgb.DMatrix(X_val.values,   label=y_val,   feature_names=FEATURES)

model_xgb = xgb.train(
    xgb_params, dtrain,
    num_boost_round=3000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=100,
    verbose_eval=200
)

model_xgb.save_model('/kaggle/working/xgboost_model.json')
cleanup(dtrain, dval)
print(f"  XGBoost saved (best round {model_xgb.best_iteration}).")

#AutoGLuon

print("\nTraining AutoGluon...")
for _name in ['predictor']:
    if _name in dir():
        try:
            exec(f'del {_name}')
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

import time, uuid
model_path = f'/kaggle/working/ag_{int(time.time())}_{uuid.uuid4().hex[:6]}/'
assert not os.path.exists(model_path), f"Path collision: {model_path}"
print(f"  Model path: {model_path}")

for entry in os.listdir('/kaggle/working/'):
    full = os.path.join('/kaggle/working/', entry)
    if entry.startswith('ag_') and os.path.isdir(full) and full != model_path:
        shutil.rmtree(full)
        print(f"  Cleaned old model dir: {entry}")

#data prep
train_ag = train_df[FEATURES + ['resistance']].copy()
val_ag   = val_df[FEATURES + ['resistance']].copy()
test_ag  = test_df[FEATURES].copy()

for col in CAT_FEATURES:
    train_ag[col] = train_ag[col].fillna('unknown').astype(str)
    val_ag[col]   = val_ag[col].fillna('unknown').astype(str)
    test_ag[col]  = test_ag[col].fillna('unknown').astype(str)

# downcast float
for col in train_ag.select_dtypes(include=['float64']).columns:
    train_ag[col] = train_ag[col].astype(np.float32)
    val_ag[col]   = val_ag[col].astype(np.float32)
    test_ag[col]  = test_ag[col].astype(np.float32)

cleanup(train_df, val_df, test_df)
print(f"  train_ag: {train_ag.shape}, val_ag: {val_ag.shape}, test_ag: {test_ag.shape}")

#fit
try:
    predictor = TabularPredictor(
        label='resistance',
        path=model_path,
        eval_metric='roc_auc',
        verbosity=2,
    ).fit(
        train_data=train_ag,
        tuning_data=val_ag,
        time_limit=3600,
        presets='best_quality',
        dynamic_stacking=False,   
    )
except Exception as e:
    print(f"AutoGluon fit failed: {e}")
    raise

#verifiying predictor
assert predictor is not None, "predictor is None after fit"
assert hasattr(predictor, '_learner') and predictor._learner.is_fit, \
    "predictor._learner is not fit — something went wrong silently"

#save preds
try:
    ag_test_preds = predictor.predict_proba(test_ag)[1].values
    ag_val_preds  = predictor.predict_proba(val_ag.drop(columns=['resistance']))[1].values
except Exception as e:
    print(f" Prediction failed: {e}")
    raise

assert len(ag_test_preds) == len(test_ag),  "test pred length mismatch"
assert len(ag_val_preds)  == len(val_ag),   "val pred length mismatch"
assert not np.isnan(ag_test_preds).any(),   "NaNs in ag_test_preds"
assert not np.isnan(ag_val_preds).any(),    "NaNs in ag_val_preds"

np.save('/kaggle/working/ag_test_preds.npy', ag_test_preds)
np.save('/kaggle/working/ag_val_preds.npy',  ag_val_preds)

#verify saved files are readable
assert np.load('/kaggle/working/ag_test_preds.npy').shape == ag_test_preds.shape
assert np.load('/kaggle/working/ag_val_preds.npy').shape  == ag_val_preds.shape

#cleanup
cleanup(train_ag, val_ag, test_ag)
del predictor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"AutoGluon done. Saved to {model_path}")
print(f"ag_test_preds shape : {ag_test_preds.shape}")
print(f"ag_val_preds  shape : {ag_val_preds.shape}")

# NGBoost

In [ ]:
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
import numpy as np
import pandas as pd
import pickle
from ngboost import NGBClassifier
from ngboost.distns import Bernoulli
from interpret.glassbox import ExplainableBoostingClassifier
from autogluon.tabular import TabularPredictor

#memory cleanup
def cleanup(*args):
    for obj in args:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020

source_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)
cleanup(source_df)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()
cleanup(source_sorted)

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})")

label_encoders = {}

X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values.astype(np.int8)
y_val   = val_df['resistance'].values.astype(np.int8)
y_test  = test_df['resistance'].values.astype(np.int8)

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('unknown').astype(str))
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_val[col]  = X_val[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    X_test[col] = X_test[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    label_encoders[col] = le

for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype(np.float32)
    X_val[col]   = X_val[col].astype(np.float32)
    X_test[col]  = X_test[col].astype(np.float32)

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight : {pos_weight:.2f}")

#ngboost
print("\nTraining NGBoost...")

sample_weights = np.where(y_train == 1, pos_weight, 1.0)

ngb_model = NGBClassifier(
    Dist=Bernoulli,
    n_estimators=1000,
    learning_rate=0.03,
    random_state=42,
    verbose=False
)

ngb_model.fit(
    X_train.values, y_train,
    X_val=X_val.values, Y_val=y_val,
    sample_weight=sample_weights,
    early_stopping_rounds=50
)

# save model
with open('/kaggle/working/ngboost_model.pkl', 'wb') as f:
    pickle.dump(ngb_model, f)

#save NGBoost uncertainty (p*(1-p)) on test set
ngb_proba_test = ngb_model.predict_proba(X_test.values)[:, 1].astype(np.float32)
ngb_var_test = (ngb_proba_test * (1 - ngb_proba_test)).astype(np.float32)
with open('/kaggle/working/ngb_var_test.pkl', 'wb') as f:
    pickle.dump(ngb_var_test, f)

print("NGBoost trained, saved, and uncertainty extracted.")

cleanup(sample_weights, ngb_proba_test, ngb_var_test)

#ebm
print("\nTraining EBM...")

ebm_model = ExplainableBoostingClassifier(
    random_state=42,
    interactions=5,
    learning_rate=0.01,
    max_bins=256,
    class_weight='balanced'
)
ebm_model.fit(X_train.values, y_train)
with open('/kaggle/working/ebm_model.pkl', 'wb') as f:
    pickle.dump(ebm_model, f)

print("EBM trained and saved.")

#autogluon

print("\nTraining AutoGluon...")


train_ag = train_df[FEATURES + ['resistance']].copy()
val_ag   = val_df[FEATURES + ['resistance']].copy()
test_ag  = test_df[FEATURES].copy()

for col in CAT_FEATURES:
    train_ag[col] = train_ag[col].fillna('unknown').astype(str)
    val_ag[col]   = val_ag[col].fillna('unknown').astype(str)
    test_ag[col]  = test_ag[col].fillna('unknown').astype(str)

predictor = TabularPredictor(
    label='resistance',
    path='/kaggle/working/autogluon_model/',
    eval_metric='roc_auc'
).fit(
    train_data=train_ag,
    tuning_data=val_ag,
    time_limit=3600,
    presets='best_quality'
)

with open('/kaggle/working/autogluon_predictor.pkl', 'wb') as f:
    pickle.dump(predictor, f)

cleanup(train_ag, val_ag, test_ag)
print("  AutoGluon trained and saved.")

print("\nTraining MLP...")

scaler_mlp = StandardScaler()
X_train_mlp = scaler_mlp.fit_transform(X_train.values.astype(np.float32))
X_val_mlp   = scaler_mlp.transform(X_val.values.astype(np.float32))
X_test_mlp  = scaler_mlp.transform(X_test.values.astype(np.float32))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  MLP device → {device}")

train_tensor = TensorDataset(torch.tensor(X_train_mlp), torch.tensor(y_train, dtype=torch.float32))
val_tensor   = TensorDataset(torch.tensor(X_val_mlp),   torch.tensor(y_val,   dtype=torch.float32))
test_tensor  = TensorDataset(torch.tensor(X_test_mlp),  torch.tensor(y_test,  dtype=torch.float32))

cleanup(X_train_mlp, X_val_mlp, X_test_mlp)

train_loader = DataLoader(train_tensor, batch_size=4096, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(val_tensor,   batch_size=4096, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_tensor,  batch_size=4096, shuffle=False, pin_memory=True)

class AMR_MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

mlp = AMR_MLP(train_tensor.tensors[0].shape[1]).to(device)
pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_loss = float('inf')
best_weights = None
patience_counter = 0
PATIENCE = 10

for epoch in range(100):
    mlp.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(mlp(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_tensor)

    mlp.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(mlp(xb), yb).item() * len(xb)
    val_loss /= len(val_tensor)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

mlp.load_state_dict(best_weights)

torch.save(best_weights, '/kaggle/working/mlp_weights.pt')
with open('/kaggle/working/scaler_mlp.pkl', 'wb') as f:
    pickle.dump(scaler_mlp, f)

cleanup(train_tensor, val_tensor, test_tensor, train_loader, val_loader, test_loader)

print("  MLP trained and saved.")

with open('/kaggle/working/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

print("\n All models and artefacts saved:")
print("   - ngboost_model.pkl")
print("   - ngb_var_test.pkl          (NGBoost uncertainty p*(1-p))")
print("   - ebm_model.pkl")
print("   - autogluon_predictor.pkl   (and folder autogluon_model/)")
print("   - mlp_weights.pt, scaler_mlp.pkl")
print("   - label_encoders.pkl")

# Comparison of Models

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.isotonic import IsotonicRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, recall_score, precision_score,
    brier_score_loss, confusion_matrix
)
from itertools import combinations
from joblib import Parallel, delayed
import pickle, os, warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

#configs

DATA_DIR = '/kaggle/input/datasets/uaisamangeldi/importmodels'
LGB_DIR  = os.path.join(DATA_DIR, 'all_lgb_files')
MLP_DIR  = os.path.join(DATA_DIR, 'all_mlp_files')
CBM_DIR  = os.path.join(DATA_DIR, 'all_cbm_files')
CSV_PATH = '/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv'

#helpers

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_bin       = (y_prob >= threshold).astype(int)
    prevalence  = y_true.mean()
    bs          = brier_score_loss(y_true, y_prob)
    bs_ref      = prevalence * (1 - prevalence)
    bss         = 1.0 - bs / bs_ref if bs_ref > 0 else np.nan
    tn, fp, fn, tp = confusion_matrix(y_true, y_bin).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return {
        'auc'         : roc_auc_score(y_true, y_prob),
        'auprc'       : average_precision_score(y_true, y_prob),
        'f1'          : f1_score(y_true, y_bin),
        'precision'   : precision_score(y_true, y_bin),
        'recall'      : recall_score(y_true, y_bin),
        'specificity' : specificity,
        'brier'       : bs,
        'bss'         : bss,
    }

def _stratified_subsample(y_true, max_samples, rng):
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    n_pos   = min(max_samples // 2, len(pos_idx))
    n_neg   = min(max_samples // 2, len(neg_idx))
    return np.concatenate([
        rng.choice(pos_idx, size=n_pos, replace=False),
        rng.choice(neg_idx, size=n_neg, replace=False)
    ])

def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, seed=42, ci=95,
                 max_samples=50_000):
    rng     = np.random.default_rng(seed)
    point   = metric_fn(y_true, y_prob)
    sub_idx = _stratified_subsample(y_true, min(max_samples, len(y_true)), rng)
    y_sub, p_sub, sub_n = y_true[sub_idx], y_prob[sub_idx], len(sub_idx)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, sub_n, sub_n)
        if len(np.unique(y_sub[idx])) < 2:
            continue
        try:
            vals.append(metric_fn(y_sub[idx], p_sub[idx]))
        except Exception:
            continue
    alpha = (100 - ci) / 2
    return (float(point),
            float(np.percentile(vals, alpha)),
            float(np.percentile(vals, 100 - alpha)))

def delong_test(y_true, y_prob_a, y_prob_b, max_samples=50_000, seed=42):
    rng = np.random.default_rng(seed)
    if len(y_true) > max_samples:
        idx      = _stratified_subsample(y_true, max_samples, rng)
        y_true   = y_true[idx]
        y_prob_a = y_prob_a[idx]
        y_prob_b = y_prob_b[idx]
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    n1, n0  = len(pos_idx), len(neg_idx)

    def placement(pos_scores, neg_scores):
        V10 = np.array([np.mean(p > neg_scores) + 0.5 * np.mean(p == neg_scores)
                        for p in pos_scores])
        V01 = np.array([np.mean(q < pos_scores) + 0.5 * np.mean(q == pos_scores)
                        for q in neg_scores])
        return V10, V01

    V10_a, V01_a = placement(y_prob_a[pos_idx], y_prob_a[neg_idx])
    V10_b, V01_b = placement(y_prob_b[pos_idx], y_prob_b[neg_idx])
    s10      = np.cov(np.stack([V10_a, V10_b]), ddof=1) / n1
    s01      = np.cov(np.stack([V01_a, V01_b]), ddof=1) / n0
    S        = s10 + s01
    var_diff = S[0,0] + S[1,1] - 2*S[0,1]
    if var_diff <= 1e-8:
        warnings.warn("DeLong var_diff near zero.", RuntimeWarning, stacklevel=2)
        return 0.0, 1.0
    z = (V10_a.mean() - V10_b.mean()) / np.sqrt(var_diff)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p)

def expected_calibration_error(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / n) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)

def hosmer_lemeshow(y_true, y_prob, n_bins=10):
    df_hl = pd.DataFrame({'y': y_true, 'p': y_prob})
    df_hl['decile'] = pd.qcut(df_hl['p'], q=n_bins, duplicates='drop')
    chi2, valid_bins = 0.0, 0
    for _, g in df_hl.groupby('decile', observed=True):
        n_g     = len(g)
        obs_pos = g['y'].sum()
        exp_pos = g['p'].sum()
        obs_neg = n_g - obs_pos
        exp_neg = n_g - exp_pos
        if exp_pos > 0 and exp_neg > 0:
            chi2 += (obs_pos - exp_pos)**2 / exp_pos
            chi2 += (obs_neg - exp_neg)**2 / exp_neg
            valid_bins += 1
    df_stat = max(valid_bins - 2, 1)
    return float(chi2), float(1 - stats.chi2.cdf(chi2, df=df_stat)), df_stat

def build_encoded_features(df, source_df, cat_features, features):
    "Label-encode categoricals using full-corpus encoders."
    X = df[features].copy()
    for col in cat_features:
        le      = LabelEncoder()
        le.fit(source_df[col].fillna('unknown').astype(str))
        mapping = dict(zip(le.classes_, le.transform(le.classes_)))
        X[col]  = X[col].fillna('unknown').astype(str).map(
            lambda s, d=mapping: d.get(s, -1))
    for col in X.select_dtypes(include=['float64']).columns:
        X[col] = X[col].astype(np.float32)
    return X

def build_catboost_features(df, cat_features, features):
    "String categoricals + float32 numerics — CatBoost."
    X = df[features].copy()
    for col in cat_features:
        X[col] = X[col].fillna('unknown').astype(str)
    for col in X.select_dtypes(include=['float64']).columns:
        X[col] = X[col].astype(np.float32)
    return X

print("Loading CSV …")
source_df = pd.read_csv(CSV_PATH).sort_values('year').reset_index(drop=True)

val_df  = source_df[(source_df['year'] > 2018) & (source_df['year'] <= 2020)].copy()
test_df = source_df[source_df['year'] > 2020].copy()

y_val  = val_df['resistance'].astype(int).values
y_test = test_df['resistance'].astype(int).values


X_val_enc  = build_encoded_features(val_df,  source_df, CAT_FEATURES, FEATURES)
X_test_enc = build_encoded_features(test_df, source_df, CAT_FEATURES, FEATURES)

X_val_cb  = build_catboost_features(val_df,  CAT_FEATURES, FEATURES)
X_test_cb = build_catboost_features(test_df, CAT_FEATURES, FEATURES)

print(f"Val  : {len(y_val):,} samples, prevalence={y_val.mean():.3f}")
print(f"Test : {len(y_test):,} samples, prevalence={y_test.mean():.3f}")

with open(os.path.join(DATA_DIR, 'lr_model.pkl'),  'rb') as f: lr_model  = pickle.load(f)
with open(os.path.join(DATA_DIR, 'scaler_lr.pkl'), 'rb') as f: scaler_lr = pickle.load(f)
with open(os.path.join(DATA_DIR, 'rf_model.pkl'),  'rb') as f: rf_model  = pickle.load(f)
with open(os.path.join(DATA_DIR, 'ngboost_model.pkl'), 'rb') as f: ngb_model = pickle.load(f)

_xgb = xgb.Booster()
_xgb.load_model(os.path.join(DATA_DIR, 'xgboost_model.json'))

from catboost import CatBoostClassifier, Pool
cbm_model = CatBoostClassifier()
cbm_model.load_model(os.path.join(CBM_DIR, 'catboost_model.cbm'))
print("All models loaded")

# =============================================================================
# BUILD val_preds
# =============================================================================
val_preds = {}

# Saved .npy (LightGBM, MLP — CatBoost handled separately below)
for key, path in [('LightGBM', os.path.join(LGB_DIR, 'lgb_val_preds.npy')),
                  ('MLP',      os.path.join(MLP_DIR, 'mlp_val_preds.npy'))]:
    if os.path.exists(path):
        val_preds[key] = np.load(path).ravel()
        print(f"{key} val preds loaded from .npy")

# CatBoost — load from .npy if available, else re-infer with Pool
cbm_val_path = os.path.join(CBM_DIR, 'catboost_val_preds.npy')
if os.path.exists(cbm_val_path):
    val_preds['CatBoost'] = np.load(cbm_val_path).ravel()
    print("CatBoost val preds loaded from .npy")
else:
    val_preds['CatBoost'] = cbm_model.predict_proba(
        Pool(X_val_cb, cat_features=CAT_FEATURES))[:, 1]
    print("✅ CatBoost val preds inferred from model")

val_preds['Logistic Regression'] = lr_model.predict_proba(
    scaler_lr.transform(X_val_enc))[:, 1]
val_preds['Random Forest'] = rf_model.predict_proba(X_val_enc)[:, 1]
val_preds['XGBoost']       = _xgb.predict(xgb.DMatrix(X_val_enc))
val_preds['NGBoost']       = ngb_model.predict_proba(X_val_enc)[:, 1]
print(" LR / RF / XGBoost / NGBoost val preds inferred")

ag_dir = os.path.join(DATA_DIR, 'autogluon_model_lean')
if os.path.isdir(ag_dir):
    from autogluon.tabular import TabularPredictor
    _ag  = TabularPredictor.load(ag_dir)
    _agv = _ag.predict_proba(val_df)
    val_preds['AutoGluon'] = _agv[1 if 1 in _agv.columns else '1'].values
    del _ag, _agv
    print("AutoGluon val preds inferred")

print(f"\nval_preds ready for : {list(val_preds.keys())}")

#build test preds
test_preds = {}

# saved .npy (LightGBM, MLP)
for key, path in [('LightGBM', os.path.join(LGB_DIR, 'lgb_test_preds.npy')),
                  ('MLP',      os.path.join(MLP_DIR, 'mlp_test_preds.npy'))]:
    if os.path.exists(path):
        test_preds[key] = np.load(path).ravel()
        print(f"✅ {key} test preds loaded from .npy")

# CatBoost — load from .npy if available, else re-infer
cbm_test_path = os.path.join(CBM_DIR, 'catboost_test_preds.npy')
if os.path.exists(cbm_test_path):
    test_preds['CatBoost'] = np.load(cbm_test_path).ravel()
    print("CatBoost test preds loaded from .npy")
else:
    test_preds['CatBoost'] = cbm_model.predict_proba(
        Pool(X_test_cb, cat_features=CAT_FEATURES))[:, 1]
    print("CatBoost test preds inferred from model")

test_preds['Logistic Regression'] = lr_model.predict_proba(
    scaler_lr.transform(X_test_enc))[:, 1]
test_preds['Random Forest'] = rf_model.predict_proba(X_test_enc)[:, 1]
test_preds['XGBoost']       = _xgb.predict(xgb.DMatrix(X_test_enc))
test_preds['NGBoost']       = ngb_model.predict_proba(X_test_enc)[:, 1]
print("LR / RF / XGBoost / NGBoost test preds inferred")

if os.path.isdir(ag_dir):
    from autogluon.tabular import TabularPredictor
    _ag  = TabularPredictor.load(ag_dir)
    _agt = _ag.predict_proba(test_df)
    test_preds['AutoGluon'] = _agt[1 if 1 in _agt.columns else '1'].values
    del _ag, _agt
    print("AutoGluon test preds inferred")

# Free memory
del X_val_enc, X_test_enc, X_val_cb, X_test_cb
del lr_model, scaler_lr, rf_model, _xgb, ngb_model, cbm_model
del val_df, test_df, source_df

model_names = list(test_preds.keys())
print(f"\ntest_preds ready for : {model_names}")

#bootstrap CI 95%
print("\n" + "="*100)
print("Per-Model Metrics with 95% Bootstrap CI {stratified, parallel}")
print("="*100)

def compute_ci_for_model(name):
    y_prob = test_preds[name]
    auc_pt,   auc_lo,   auc_hi   = bootstrap_ci(y_test, y_prob, roc_auc_score)
    auprc_pt, auprc_lo, auprc_hi = bootstrap_ci(y_test, y_prob, average_precision_score)
    brier_pt, brier_lo, brier_hi = bootstrap_ci(y_test, y_prob, brier_score_loss)
    return {
        'Model' : name,
        'AUC'   : f"{auc_pt:.4f} [{auc_lo:.4f}–{auc_hi:.4f}]",
        'AUPRC' : f"{auprc_pt:.4f} [{auprc_lo:.4f}–{auprc_hi:.4f}]",
        'Brier' : f"{brier_pt:.4f} [{brier_lo:.4f}–{brier_hi:.4f}]",
    }

ci_rows = Parallel(n_jobs=-1)(
    delayed(compute_ci_for_model)(name) for name in model_names)
print(pd.DataFrame(ci_rows).to_string(index=False))

#DeLong pairwise (stratified, parallel, Bonferroni)

print("\n" + "="*100)
print("DeLong Pairwise AUC Tests {stratified, Bonferroni corrected}")
print("="*100)

pairs = list(combinations(model_names, 2))

def delong_for_pair(a, b):
    z, p = delong_test(y_test, test_preds[a], test_preds[b])
    return {'Model A': a, 'Model B': b, 'Z': round(z,4), 'p-value': round(p,4), '_raw_p': p}

results_pairs = Parallel(n_jobs=-1)(
    delayed(delong_for_pair)(a, b) for a, b in pairs)

delong_rows, raw_pvals = [], []
for res in results_pairs:
    raw_pvals.append(res.pop('_raw_p'))
    delong_rows.append(res)

n_tests    = len(raw_pvals)
alpha_corr = 0.05 / n_tests
for i, row in enumerate(delong_rows):
    row['p-adj (Bonf)'] = round(min(raw_pvals[i] * n_tests, 1.0), 4)
    row['Significant']  = 'OK' if raw_pvals[i] < alpha_corr else 'ERROR'

print(f"Bonferroni α = 0.05 / {n_tests} = {alpha_corr:.4f}\n")
print(pd.DataFrame(delong_rows).to_string(index=False))


#Calibration (ECE primary, HL indicative)
print("\n" + "="*100)
print("Calibration: ECE + Hosmer-Lemeshow")
print("="*100)

cal_rows = []
for name in model_names:
    y_prob         = test_preds[name]
    ece            = expected_calibration_error(y_test, y_prob)
    chi2, p_hl, df = hosmer_lemeshow(y_test, y_prob)
    cal_rows.append({
        'Model'        : name,
        'ECE'          : round(ece, 4),
        'HL χ²'        : round(chi2, 3),
        'HL df'        : df,
        'HL p-value'   : round(p_hl, 4),
        'HL (N=20M)' : 'OK' if p_hl > 0.05 else 'ERROR (expected lowk)',
    })

print(pd.DataFrame(cal_rows).sort_values('ECE').to_string(index=False))


#Isotonic recalibration (fit on val, predict on test)

print("\n" + "="*100)
print("Isotonic Regression Recalibration (val -> test)")
print("="*100)

isotonic_rows    = []
test_preds_recal = {}

for name in model_names:
    if name not in val_preds:
        print(f"{name}: no val preds — skipping")
        test_preds_recal[name] = test_preds[name]
        continue

    n_val = len(val_preds[name])
    if n_val < 10_000:
        print(f"{name}: val set only {n_val:,} samples — interpret ΔECE cautiously")

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(val_preds[name], y_val)
    recal_probs            = iso.predict(test_preds[name])  # ← predict on TEST
    test_preds_recal[name] = recal_probs

    isotonic_rows.append({
        'Model'      : name,
        'Val N'      : f"{n_val:,}",
        'ECE before' : round(expected_calibration_error(y_test, test_preds[name]), 4),
        'ECE after'  : round(expected_calibration_error(y_test, recal_probs), 4),
        'ΔECE'       : round(expected_calibration_error(y_test, recal_probs) -
                             expected_calibration_error(y_test, test_preds[name]), 4),
        'AUC before' : round(roc_auc_score(y_test, test_preds[name]), 4),
        'AUC after'  : round(roc_auc_score(y_test, recal_probs), 4),
    })

print(pd.DataFrame(isotonic_rows).sort_values('ECE after').to_string(index=False))
print("\n Negative difference in ECE = improved calibration after isotonic recalibration")

#NGBOOST UNCERTAINTY
if 'NGBoost' in test_preds:
    print("\n" + "="*100)
    print("NGBoost Per-Isolate Predictive Uncertainty [p(1−p)]")
    print("="*100)

    p           = test_preds['NGBoost']
    uncertainty = p * (1 - p)

    print(f"\n  Uncertainty summary across {len(p):,} isolates:")
    print(f"  Mean   : {uncertainty.mean():.4f}")
    print(f"  Median : {np.median(uncertainty):.4f}")
    print(f"  Std    : {uncertainty.std():.4f}")
    print(f"  Min    : {uncertainty.min():.4f}")
    print(f"  Max    : {uncertainty.max():.4f}")

    q1, q2, q3 = np.percentile(uncertainty, [25, 50, 75])
    print(f"\n  Quartiles  Q1={q1:.4f}  Q2={q2:.4f}  Q3={q3:.4f}")
    thresh_90 = np.percentile(uncertainty, 90)
    print(f"\n  High-uncertainty isolates (top 10%, u ≥ {thresh_90:.4f}): "
          f"{(uncertainty >= thresh_90).sum():,}")
    print("Flag these isolates for expert review in clinical use")


print("\n" + "="*100)
print("Full Metrics Summary (sorted by AUPRC)")
print("="*100)

full_rows = []
for name in model_names:
    m = compute_metrics(y_test, test_preds[name])
    m['Model'] = name
    full_rows.append(m)

df_full = pd.DataFrame(full_rows)[[
    'Model','auprc','auc','f1','precision','recall','specificity','brier','bss'
]].round(4).sort_values('auprc', ascending=False).reset_index(drop=True)

print(df_full.to_string(index=False))
print("="*100)
print(f"\nModels evaluated: {len(df_full)}")

In [ ]:
import numpy as np
import pandas as pd
import pickle, os
import xgboost as xgb
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
DATA_DIR = '/kaggle/input/datasets/uaisamangeldi/importmodels'
LGB_DIR  = os.path.join(DATA_DIR, 'all_lgb_files')
MLP_DIR  = os.path.join(DATA_DIR, 'all_mlp_files')
CBM_DIR  = os.path.join(DATA_DIR, 'all_cbm_files')
CSV_PATH = '/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv'

# =============================================================================
# LOAD TEST DATA
# =============================================================================
source_df = pd.read_csv(CSV_PATH).sort_values('year').reset_index(drop=True)
test_df   = source_df[source_df['year'] > 2020].copy()
y_test    = test_df['resistance'].astype(int).values

# Label-encoded X_test for sklearn/xgb models
X_test_enc = test_df[FEATURES].copy()
for col in CAT_FEATURES:
    le       = LabelEncoder()
    full_col = source_df[col].fillna('unknown').astype(str)
    le.fit(full_col)
    mapping  = dict(zip(le.classes_, le.transform(le.classes_)))
    X_test_enc[col] = X_test_enc[col].fillna('unknown').astype(str).map(
        lambda s, d=mapping: d.get(s, -1))
for col in X_test_enc.select_dtypes(include=['float64']).columns:
    X_test_enc[col] = X_test_enc[col].astype(np.float32)

del source_df
print(f"Test set: {len(y_test):,} samples, prevalence={y_test.mean():.3f}")

# =============================================================================
# REBUILD test_preds FOR TOP 3 MODELS ONLY
# =============================================================================
test_preds = {}

# LightGBM — saved preds
lgb_path = os.path.join(LGB_DIR, 'lgb_test_preds.npy')
if os.path.exists(lgb_path):
    test_preds['LightGBM'] = np.load(lgb_path).ravel()
    print("✅ LightGBM loaded")

# CatBoost — reload model with raw string features
cbm_path = os.path.join(CBM_DIR, 'catboost_model.cbm')
if os.path.exists(cbm_path):
    from catboost import CatBoostClassifier, Pool
    cbm_model   = CatBoostClassifier()
    cbm_model.load_model(cbm_path)
    X_test_cb   = test_df[FEATURES].copy()
    for col in CAT_FEATURES:
        X_test_cb[col] = X_test_cb[col].fillna('unknown').astype(str)
    for col in X_test_cb.select_dtypes(include=['float64']).columns:
        X_test_cb[col] = X_test_cb[col].astype(np.float32)
    test_preds['CatBoost'] = cbm_model.predict_proba(
        Pool(X_test_cb, cat_features=CAT_FEATURES))[:, 1]
    del cbm_model, X_test_cb
    print("✅ CatBoost loaded")

# AutoGluon — needs raw test_df
ag_dir = os.path.join(DATA_DIR, 'autogluon_model_lean')
if os.path.isdir(ag_dir):
    from autogluon.tabular import TabularPredictor
    predictor  = TabularPredictor.load(ag_dir)
    ag_preds   = predictor.predict_proba(test_df)
    col        = 1 if 1 in ag_preds.columns else '1'
    test_preds['AutoGluon'] = ag_preds[col].values
    del predictor
    print("✅ AutoGluon loaded")

# =============================================================================
# CONFUSION MATRICES
# =============================================================================
top_models = ['AutoGluon', 'CatBoost', 'LightGBM']
fig, axes  = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, top_models):
    if name not in test_preds:
        ax.set_visible(False)
        continue

    y_prob = test_preds[name]
    y_pred = (y_prob >= 0.5).astype(int)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(cm, display_labels=['Susceptible', 'Resistant'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=13, fontweight='bold')

    tn, fp, fn, tp = cm.ravel()
    ax.set_xlabel(
        f"Sensitivity={tp/(tp+fn):.3f}  Specificity={tn/(tn+fp):.3f}\n"
        f"PPV={tp/(tp+fp):.3f}  NPV={tn/(tn+fn):.3f}",
        fontsize=9)

plt.suptitle('Confusion Matrices — Test Set (year > 2020)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved to /kaggle/working/confusion_matrices.png")

# Applying Model

In [ ]:
SANITATION_PATH    = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874.csv'
WATER_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137.csv'
HOSPITAL_BEDS_PATH = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206.csv'
TEMPERATURE_PATH   = '/kaggle/input/datasets/uaisamangeldi/highfeature/average-monthly-surface-temperature/average-monthly-surface-temperature.csv'
AWARE_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/data.csv'

In [ ]:
import gc
KZ_MACRO_FILES = {
    'gdp_per_capita_usd':         '/kaggle/input/datasets/uaisamangeldi/macro-kz/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv',
    'health_expenditure_pct_gdp': '/kaggle/input/datasets/uaisamangeldi/macro-kz/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_558/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_558.csv',
    'population_total':           '/kaggle/input/datasets/uaisamangeldi/macro-kz/API_SP.POP.TOTL_DS2_en_csv_v2_61/API_SP.POP.TOTL_DS2_en_csv_v2_61.csv',
}
KZ_DID_PATH = '/kaggle/input/datasets/uaisamangeldi/kz-data/kazakhstan_antibiotics_did_2017_2023.csv'
KZ_AREA_KM2 = 2_724_900

# ── KZ AWaRe Access % (hardcoded from table) ─────────────────────────────────
KZ_AWARE = pd.DataFrame({
    'year':             [2017,  2018,  2019,  2020,  2021,  2022,  2023],
    'aware_access_pct': [33.00, 15.00, 52.21, 40.63, 40.27, 43.17, 44.33],
})

# ── KZ New WB Features (sanitation, water, hospital beds) ────────────────────
# Pull from the same global WB files used in Cell 4
KZ_SANITATION_PATH    = SANITATION_PATH
KZ_WATER_PATH         = WATER_PATH
KZ_HOSPITAL_BEDS_PATH = HOSPITAL_BEDS_PATH
KZ_TEMPERATURE_PATH   = TEMPERATURE_PATH

def load_kz_wb_indicator(path, col_name):
    df = pd.read_csv(path, skiprows=4)
    df.columns = df.columns.str.strip()
    df = df[df['Country Name'] == 'Kazakhstan'].copy()
    year_cols = [c for c in df.columns if c.strip().isdigit() and int(c.strip()) >= 2000]
    df = df.melt(id_vars=['Country Name'], value_vars=year_cols, var_name='year', value_name=col_name)
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df = df.dropna(subset=['year', col_name])
    df['year'] = df['year'].astype(int)
    return df[['year', col_name]]

# ── World Bank macros ─────────────────────────────────────────────────────────
kz_macro = load_kz_wb_indicator(KZ_MACRO_FILES['gdp_per_capita_usd'], 'gdp_per_capita_usd')
for col, path in list(KZ_MACRO_FILES.items())[1:]:
    kz_macro = kz_macro.merge(load_kz_wb_indicator(path, col), on='year', how='outer')

kz_macro = kz_macro.sort_values('year').reset_index(drop=True)
kz_macro['population_density_per_sq_km'] = kz_macro['population_total'] / KZ_AREA_KM2

# ── Merge new WB features into kz_macro ──────────────────────────────────────
for path, name in [
    (KZ_SANITATION_PATH,    'sanitation_access_pct'),
    (KZ_WATER_PATH,         'water_access_pct'),
    (KZ_HOSPITAL_BEDS_PATH, 'hospital_beds_per_1000'),
]:
    try:
        df_tmp = load_kz_wb_indicator(path, name)
        kz_macro = kz_macro.merge(df_tmp, on='year', how='left')
        print(f" {name}: OK")
    except Exception as e:
        print(f" {name}: FAILED ({e})")
        kz_macro[name] = np.nan

# ── Merge KZ temperature ──────────────────────────────────────────────────────
try:
    df_temp = pd.read_csv(KZ_TEMPERATURE_PATH, usecols=['Entity', 'Day', 'Monthly average'])
    df_temp = df_temp[df_temp['Entity'] == 'Kazakhstan'].copy()
    df_temp['year'] = pd.to_datetime(df_temp['Day'], errors='coerce').dt.year
    kz_temp_yearly = (df_temp.groupby('year', as_index=False)['Monthly average']
                              .mean()
                              .rename(columns={'Monthly average': 'mean_temp_celsius'}))
    del df_temp; gc.collect()
    kz_macro = kz_macro.merge(kz_temp_yearly, on='year', how='left')
    print(" mean_temp_celsius: OK")
except Exception as e:
    print(f" mean_temp_celsius: FAILED ({e})")
    kz_macro['mean_temp_celsius'] = np.nan

# ── Merge AWaRe ───────────────────────────────────────────────────────────────
kz_macro = kz_macro.merge(KZ_AWARE, on='year', how='left')

# ── Load KZ DID ───────────────────────────────────────────────────────────────
kz_did = pd.read_csv(KZ_DID_PATH)
kz_did.columns = kz_did.columns.str.strip()
kz_did['antibiotic']     = kz_did['antibiotic'].str.strip().str.lower().str.replace('/', ' / ', regex=False)
kz_did['aware_category'] = kz_did['aware_category'].fillna('Unknown').str.strip()
kz_did['did']            = pd.to_numeric(kz_did['did'], errors='coerce').fillna(0)
kz_did['year']           = kz_did['year'].astype(int)

# ── Per-class and total consumption ──────────────────────────────────────────
kz_class_did = (
    kz_did
    .groupby(['year', 'aware_category'])['did']
    .sum()
    .reset_index()
    .rename(columns={'aware_category': 'antibiotic_class',
                     'did': 'consumption_did_per_class'})
)

kz_total_did = (
    kz_did
    .groupby('year')['did']
    .sum()
    .reset_index()
    .rename(columns={'did': 'consumption_ddd_total'})
)

# ── Filter macro to 2017–2023 and merge consumption ──────────────────────────
kz_macro_filtered = kz_macro[(kz_macro['year'] >= 2017) & (kz_macro['year'] <= 2023)].copy()
kz_macro_filtered = kz_macro_filtered.merge(kz_total_did, on='year', how='left')

# ── Impute missing new features using global means ────────────────────────────
new_feature_cols = ['sanitation_access_pct', 'water_access_pct',
                    'hospital_beds_per_1000', 'mean_temp_celsius', 'aware_access_pct']
for col in new_feature_cols:
    if col in kz_macro_filtered.columns:
        if kz_macro_filtered[col].isna().any():
            global_mean = full_df[col].mean() if col in full_df.columns else np.nan
            kz_macro_filtered[col] = kz_macro_filtered[col].fillna(global_mean)
            print(f" {col}: filled {kz_macro_filtered[col].isna().sum()} NaNs with global mean ({global_mean:.2f})")

# ── Build final KZ dataframe ──────────────────────────────────────────────────
kz_final = kz_did.merge(kz_macro_filtered, on='year', how='inner')
kz_final = kz_final.rename(columns={'aware_category': 'antibiotic_class'})
kz_final = kz_final.merge(kz_class_did, on=['year', 'antibiotic_class'], how='left')

# Fallback imputation for class-level DID
global_mean_did = np.expm1(full_df['log_consumption_did_per_class'].mean())
kz_final['consumption_did_per_class'] = kz_final['consumption_did_per_class'].fillna(global_mean_did)

# ── Log transforms ────────────────────────────────────────────────────────────
kz_final['log_gdp_per_capita']            = np.log1p(kz_final['gdp_per_capita_usd'].clip(lower=0))
kz_final['log_consumption_ddd']           = np.log1p(kz_final['consumption_ddd_total'].clip(lower=0))
kz_final['log_consumption_did_per_class'] = np.log1p(kz_final['consumption_did_per_class'].clip(lower=0))

# ── Drop raw columns ──────────────────────────────────────────────────────────
kz_final = kz_final.drop(columns=['gdp_per_capita_usd', 'consumption_ddd_total',
                                   'consumption_did_per_class', 'population_total'])
kz_final = kz_final.sort_values(['year', 'antibiotic']).reset_index(drop=True)

# ── Verify ────────────────────────────────────────────────────────────────────
log_cols = ['log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class']
check_cols = log_cols + new_feature_cols
print(f"\nKazakhstan inference dataset: {kz_final.shape}")
print(f"\nFeature coverage (non-null %):")
print((kz_final[check_cols].notna().mean() * 100).round(1))
print(f"\nSample:")
print(kz_final[['year', 'antibiotic', 'antibiotic_class'] + log_cols].head(6).to_string(index=False))

kz_final.to_csv('/kaggle/working/kazakhstan_final_prediction_input_2017_2023.csv', index=False)
print("\nSaved.")

In [ ]:
kz_df = kz_final

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
import warnings

# =============================================================================
# POOL HELPERS – defined early so they are available
# =============================================================================
def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    for col in X.select_dtypes(include='float64').columns:
        X[col] = X[col].astype(np.float32)
    if has_labels:
        return Pool(
            data=X,
            label=df['resistance'].values.astype(np.int8),
            cat_features=CAT_FEATURES
        )
    return Pool(data=X, cat_features=CAT_FEATURES)


def make_kz_pool(df):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    for col in X.select_dtypes(include='float64').columns:
        X[col] = X[col].astype(np.float32)
    return Pool(data=X, cat_features=CAT_FEATURES)


# =============================================================================
# BUILD KZ INFERENCE DATAFRAME
# Cross every KZ antibiotic‑year row with 6 target organisms
# =============================================================================
TARGET_ORGANISMS = [
    'Escherichia coli',
    'Klebsiella pneumoniae',
    'Staphylococcus aureus',
    'Pseudomonas aeruginosa',
    'Acinetobacter baumannii',
    'Enterococcus faecium',
]

prediction_rows = []
for _, row in kz_input.iterrows():
    for org in TARGET_ORGANISMS:
        new_row             = row.to_dict()
        new_row['organism'] = org
        new_row['country']  = 'Kazakhstan'
        prediction_rows.append(new_row)

kz_df = pd.DataFrame(prediction_rows)
kz_df = kz_df.loc[:, ~kz_df.columns.duplicated()]
print(f"\nKZ inference rows (before filtering): {len(kz_df):,}")

# =============================================================================
# CLINICAL PLAUSIBILITY FILTER (EUCAST‑derived)
# Only combinations for which a clinical breakpoint exists are kept.
# =============================================================================
GRAM_POSITIVE = {
    'Staphylococcus aureus',
    'Enterococcus faecium',
    'Streptococcus pneumoniae',
    'Enterococcus faecalis',
}

GRAM_NEGATIVE = {
    'Escherichia coli',
    'Klebsiella pneumoniae',
    'Pseudomonas aeruginosa',
    'Acinetobacter baumannii',
    'Enterobacter cloacae',
    'Proteus mirabilis',
}

VALID_COMBINATIONS = {
    # Carbapenems — gram-negative only
    'Meropenem':                 GRAM_NEGATIVE,
    'Imipenem':                  GRAM_NEGATIVE,
    'Ertapenem':                 GRAM_NEGATIVE - {'Pseudomonas aeruginosa', 'Acinetobacter baumannii'},
    # Penicillins
    'Ampicillin':                {'Escherichia coli', 'Enterococcus faecium', 'Enterococcus faecalis'},
    'Amoxicillin-clavulanate':   GRAM_NEGATIVE - {'Pseudomonas aeruginosa', 'Acinetobacter baumannii'},
    'Piperacillin-tazobactam':   GRAM_NEGATIVE,
    # Cephalosporins — gram-negative focused
    'Ceftriaxone':               GRAM_NEGATIVE - {'Pseudomonas aeruginosa', 'Acinetobacter baumannii'},
    'Ceftazidime':               GRAM_NEGATIVE,
    'Cefepime':                  GRAM_NEGATIVE,
    # Fluoroquinolones — broad but not Enterococcus
    'Ciprofloxacin':             (GRAM_NEGATIVE | {'Staphylococcus aureus'}) - {'Enterococcus faecium'},
    'Levofloxacin':              GRAM_NEGATIVE | {'Staphylococcus aureus'},
    # Aminoglycosides
    'Gentamicin':                GRAM_NEGATIVE | {'Staphylococcus aureus'},
    'Amikacin':                  GRAM_NEGATIVE,
    'Tobramycin':                GRAM_NEGATIVE,
    # Glycopeptides — gram‑positive ONLY
    'Vancomycin':                GRAM_POSITIVE,
    'Teicoplanin':               GRAM_POSITIVE,
    # Polymyxins — gram-negative ONLY
    'Colistin':                  GRAM_NEGATIVE - {'Proteus mirabilis'},
    'Polymyxin B':               GRAM_NEGATIVE,
    # Anti‑MRSA / gram‑positive only
    'Linezolid':                 GRAM_POSITIVE,
    'Daptomycin':                GRAM_POSITIVE,
    'Oxacillin':                 {'Staphylococcus aureus'},
    'Clindamycin':               GRAM_POSITIVE,
    'Rifampicin':                GRAM_POSITIVE,
    # Tetracyclines
    'Tetracycline':              GRAM_POSITIVE | {'Escherichia coli'},
    'Tigecycline':               (GRAM_NEGATIVE | GRAM_POSITIVE) - {'Pseudomonas aeruginosa'},
    'Doxycycline':               GRAM_POSITIVE | {'Escherichia coli'},
    # Trimethoprim
    'Trimethoprim-sulfamethoxazole': GRAM_NEGATIVE | {'Staphylococcus aureus'},
}


def is_valid_combination(antibiotic: str, organism: str) -> bool:
    """
    Returns True if a clinical EUCAST breakpoint exists for this pair.
    Unknown antibiotics default to True – we keep them.
    """
    valid_orgs = VALID_COMBINATIONS.get(antibiotic)
    if valid_orgs is None:
        return True
    return organism in valid_orgs


before = len(kz_df)
kz_df['valid_combination'] = kz_df.apply(
    lambda r: is_valid_combination(r['antibiotic'], r['organism']),
    axis=1,
)
invalid_df = kz_df[~kz_df['valid_combination']]
kz_df      = kz_df[kz_df['valid_combination']].drop(columns='valid_combination')
after = len(kz_df)

print(f"\n─ Clinical plausibility filter ─")
print(f"  Rows before  : {before:,}")
print(f"  Rows removed : {before - after:,}  ({(before-after)/before:.1%})")
print(f"  Rows after   : {after:,}")

if len(invalid_df) > 0:
    print("\nTop 10 removed combinations:")
    top_invalid = (
        invalid_df.groupby(['antibiotic', 'organism'])
        .size()
        .reset_index(name='n')
        .sort_values('n', ascending=False)
        .head(10)
    )
    print(top_invalid.to_string(index=False))

# =============================================================================
# Verify all required features are present in the filtered dataframe
# =============================================================================
missing_feats = [f for f in FEATURES if f not in kz_df.columns]
if missing_feats:
    print(f"\n⚠️  Missing features in KZ data: {missing_feats}")
else:
    print("\n✅ All features present in KZ data")

# =============================================================================
# Create the KZ prediction pool (with filtered data)
# =============================================================================
kz_pool_fixed = make_kz_pool(kz_df)

print("✅ KZ prediction pool built on clinically valid rows.\n")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

feature_cols = [
    'log_gdp',          # economic capacity
    'health_exp',       # healthcare investment
    'pop_density',      # transmission risk
    'log_cons_total',   # overall antibiotic pressure
    'log_cons_per_class', # class-specific AMR pressure
    'sanitation',       # infection prevention
    'water',            # infection prevention
    'hospital_beds',    # healthcare capacity
    'mean_temp',        
    'aware_access',     
]

# Domain-justified weights:
# Higher weight = more AMR-relevant signal
# Lower weight = infrastructure proxies that distort KZ placement
feature_weights = {
    'log_gdp':            1.0,
    'health_exp':         1.2,   # was 1.5
    'pop_density':        0.9,   # was 0.8
    'log_cons_total':     1.5,   # was 2.0
    'log_cons_per_class': 1.5,   # was 2.0
    'sanitation':         0.8,   # was 0.7
    'water':              0.8,   # was 0.5  ← too aggressive before
    'hospital_beds':      0.8,   # was 0.5  ← too aggressive before
    'mean_temp':          0.9,   # was 0.8
    'aware_access':       1.2,   # was 1.5
}
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(country_profile[feature_cols])

# Apply weights after scaling so they're on the same scale
weights   = np.array([feature_weights[f] for f in feature_cols])
X_cluster = X_scaled * weights

kmeans = KMeans(n_clusters=6, random_state=42, n_init=20)
country_profile['cluster'] = kmeans.fit_predict(X_cluster)

kz_cluster = country_profile.loc['Kazakhstan', 'cluster']
print(f"Kazakhstan is in cluster {kz_cluster}")

print("\nCountries in Kazakhstan's cluster:")
cluster_members = country_profile[country_profile['cluster'] == kz_cluster].drop(columns='cluster')
print(cluster_members.index.tolist())

print("\nPost-Soviet cluster assignments:")
check = ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia']
ps_df = country_profile[country_profile.index.isin(check)][['cluster']]
print(ps_df.to_string())

# ── Distance to centroid using weighted space ─────────────────────────────────
centroid   = kmeans.cluster_centers_[kz_cluster]
distances  = {}
for country in cluster_members.index:
    row    = country_profile.loc[country, feature_cols].values.reshape(1, -1)
    scaled = scaler.transform(row) * weights
    distances[country] = np.linalg.norm(scaled - centroid)

dist_df = pd.Series(distances).sort_values().rename('distance_to_centroid')
print("\nDistance to cluster centroid (lower = more similar):")
print(dist_df.to_string())

country_profile.to_csv('/kaggle/working/country_clusters.csv')
print("\n✅ Saved country_clusters.csv")

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings('ignore')

print("=== FIXED LOCO — CatBoost (per-group retraining) ===\n")

# =============================================================================
# FEATURE SET (matches your notebook exactly)
# =============================================================================
CAT_FEATURES = ['organism', 'antibiotic', 'antibiotic_class']
NUM_FEATURES = [
    'year',
    'log_gdp_per_capita',
    'health_expenditure_pct_gdp',
    'population_density_per_sq_km',
    'log_consumption_ddd',
    'log_consumption_did_per_class',
    'sanitation_access_pct',
    'water_access_pct',
    'hospital_beds_per_1000',
    'mean_temp_celsius',
    'aware_access_pct',
]
FEATURES = CAT_FEATURES + NUM_FEATURES

# =============================================================================
# CONFIG
# =============================================================================
TRAIN_CUTOFF    = 2018          
ES_VAL_YEARS    = [2019, 2020]  
KZ_INFER_YEARS  = list(range(2017, 2024))

EXCLUDE_FROM_TRAINING = ['Kazakhstan']

CSV_PATH = '/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv'
KZ_PATH  = '/kaggle/input/datasets/uaisamangeldi/kzzzzz/kz_inference_filtered.csv'
OUT_DIR  = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# =============================================================================
# LOAD DATA
# =============================================================================
print("Loading data")
full_df  = pd.read_csv(CSV_PATH, low_memory=False)
kz_input = pd.read_csv(KZ_PATH)

print(f"  Global rows : {len(full_df):,}")
print(f"  KZ input    : {len(kz_input):,} rows")
print(f"  Countries   : {full_df['country'].nunique()}")

# =============================================================================
# BUILD KZ INFERENCE DATAFRAME
# Cross every KZ antibiotic-year row with 6 target organisms
# =============================================================================
TARGET_ORGANISMS = [
    'Escherichia coli',
    'Klebsiella pneumoniae',
    'Staphylococcus aureus',
    'Pseudomonas aeruginosa',
    'Acinetobacter baumannii',
    'Enterococcus faecium',
]

prediction_rows = []
for _, row in kz_input.iterrows():
    for org in TARGET_ORGANISMS:
        new_row             = row.to_dict()
        new_row['organism'] = org
        new_row['country']  = 'Kazakhstan'
        prediction_rows.append(new_row)

kz_df = pd.DataFrame(prediction_rows)
kz_df = kz_df.loc[:, ~kz_df.columns.duplicated()]
print(f"\nKZ inference rows (antibiotic × year × organism): {len(kz_df):,}")

# Verify all required features are present in kz_df
missing_feats = [f for f in FEATURES if f not in kz_df.columns]
if missing_feats:
    print(f"Missing features in KZ data: {missing_feats}")
else:
    print("All features present in KZ data")

# =============================================================================
# LOCO REFERENCE GROUPS
# =============================================================================
all_countries = full_df['country'].unique().tolist()

loco_groups = {
    'post_soviet': [
        'Russia', 'Ukraine', 'Belarus',
        'Lithuania', 'Latvia', 'Estonia'
    ],
    'structural_peers': [
        'Qatar', 'Turkey', 'Romania',
        'Bulgaria', 'Hungary', 'Poland'
    ],
    'all_except_kz': [
        c for c in all_countries
        if c not in EXCLUDE_FROM_TRAINING
    ],
}

# =============================================================================
# POOL HELPERS
# =============================================================================
def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    for col in X.select_dtypes(include='float64').columns:
        X[col] = X[col].astype(np.float32)
    if has_labels:
        return Pool(
            data=X,
            label=df['resistance'].values.astype(np.int8),
            cat_features=CAT_FEATURES
        )
    return Pool(data=X, cat_features=CAT_FEATURES)


def make_kz_pool(df):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    for col in X.select_dtypes(include='float64').columns:
        X[col] = X[col].astype(np.float32)
    return Pool(data=X, cat_features=CAT_FEATURES)

# =============================================================================
# CATBOOST BASE PARAMS
# =============================================================================
def get_catboost_params(scale_pos_weight, iterations, early_stopping=100):
    return {
        'learning_rate'        : 0.03,
        'depth'                : 7,
        'l2_leaf_reg'          : 3,
        'eval_metric'          : 'AUC',
        'random_seed'          : 42,
        'verbose'              : 200,
        'early_stopping_rounds': early_stopping,
        'iterations'           : iterations,
        'task_type'            : 'GPU',
        'scale_pos_weight'     : scale_pos_weight,
    }

# =============================================================================
# FIXED LOCO LOOP
# =============================================================================
loco_results  = {}
kz_pool_fixed = make_kz_pool(kz_df)  # built once, reused

for group_name, countries in loco_groups.items():
    print(f"\n{'='*70}")
    print(f"[{group_name.upper()}]")

    # ── Filter to available countries (some may not be in dataset) ────────────
    available = [c for c in countries if c in full_df['country'].unique()]
    missing   = set(countries) - set(available)
    if missing:
        print(f"not in dataset: {sorted(missing)}")
    if not available:
        print("  Skipped — no countries found")
        continue

    # ── Isolate this group's data, always exclude KZ ─────────────────────────
    group_df = full_df[
        full_df['country'].isin(available)
    ].copy()

    # ── Temporal split (FIXED: train ≤2018, val = 2019-2020) ─────────────────
    es_train_df = group_df[
        group_df['year'] <= TRAIN_CUTOFF
    ].copy()

    es_val_df = group_df[
        group_df['year'].isin(ES_VAL_YEARS)
    ].copy()

    print(f"  Countries   : {len(available)} {available[:5]}{'...' if len(available)>5 else ''}")
    print(f"  Train rows  : {len(es_train_df):,}  (≤{TRAIN_CUTOFF})")
    print(f"  Val rows    : {len(es_val_df):,}   (years {ES_VAL_YEARS})")

    if len(es_val_df) < 200:
        print(f"  Skipped — val set too small ({len(es_val_df)} rows)")
        continue

    if len(es_train_df) < 500:
        print(f"  Skipped — train set too small ({len(es_train_df)} rows)")
        continue

    # ── Compute scale_pos_weight from THIS group's training data ─────────────
    y_tr = es_train_df['resistance'].values
    scale_pos_weight = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    print(f"  scale_pos_weight : {scale_pos_weight:.4f}")

    # ── STAGE 1: Find best iteration via early stopping ───────────────────────
    print(f"\n  Stage 1 — Finding best iteration...")
    params_stage1 = get_catboost_params(
        scale_pos_weight=scale_pos_weight,
        iterations=5000,
        early_stopping=100
    )

    cal_model = CatBoostClassifier(**params_stage1)
    cal_model.fit(
        make_pool(es_train_df),
        eval_set=make_pool(es_val_df),
        use_best_model=True,
    )
    best_iter = cal_model.best_iteration_
    print(f"  Best iteration   : {best_iter}")
    del cal_model
    gc.collect()

    # ── STAGE 2: Retrain on full ≤2018 data at best_iter ─────────────────────
    print(f"\n  Stage 2 — Final retrain on all ≤{TRAIN_CUTOFF} data...")
    params_stage2 = get_catboost_params(
        scale_pos_weight=scale_pos_weight,
        iterations=best_iter,
        early_stopping=None,
    )
    params_stage2['verbose'] = 100

    loco_model = CatBoostClassifier(**params_stage2)
    loco_model.fit(make_pool(es_train_df))

    # ── Save model ────────────────────────────────────────────────────────────
    model_path = f'{OUT_DIR}/loco_{group_name}.cbm'
    loco_model.save_model(model_path)
    print(f"  Saved → {model_path}")

    # ── Evaluate on truly held-out 2019-2020 ─────────────────────────────────
    val_proba   = loco_model.predict_proba(make_pool(es_val_df))[:, 1]
    y_val       = es_val_df['resistance'].values

    val_auc     = roc_auc_score(y_val, val_proba)   if y_val.std() > 0 else np.nan
    val_auprc   = average_precision_score(y_val, val_proba) if y_val.std() > 0 else np.nan
    val_brier   = brier_score_loss(y_val, val_proba)

    print(f"\n  ── Val Metrics (2019-2020, truly held-out) ──")
    print(f"  AUC   : {val_auc:.4f}")
    print(f"  AUPRC : {val_auprc:.4f}")
    print(f"  Brier : {val_brier:.4f}")

    # ── Isotonic calibration (fit on val, apply to KZ) ───────────────────────
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(val_proba, y_val)

    val_proba_cal  = calibrator.predict(val_proba)
    val_brier_cal  = brier_score_loss(y_val, val_proba_cal)
    print(f"\n  ── After Isotonic Calibration ──")
    print(f"  Brier (raw) : {val_brier:.4f}")
    print(f"  Brier (cal) : {val_brier_cal:.4f}  ({'improved' if val_brier_cal < val_brier else 'no improvement'})")

    # ── Predict Kazakhstan ────────────────────────────────────────────────────
    kz_proba_raw = loco_model.predict_proba(kz_pool_fixed)[:, 1]
    kz_proba_cal = calibrator.predict(kz_proba_raw)

    print(f"\n  ── KZ Predictions ──")
    print(f"  KZ mean prob (raw) : {kz_proba_raw.mean():.4f}")
    print(f"  KZ mean prob (cal) : {kz_proba_cal.mean():.4f}")
    print(f"  KZ % resistant     : {(kz_proba_cal >= 0.5).mean():.1%}")

    # ── Store results ─────────────────────────────────────────────────────────
    loco_results[group_name] = {
        'kz_proba_raw' : kz_proba_raw,
        'kz_proba_cal' : kz_proba_cal,        # use this for ensemble
        'val_auc'      : val_auc,
        'val_auprc'    : val_auprc,
        'val_brier_raw': val_brier,
        'val_brier_cal': val_brier_cal,
        'best_iter'    : best_iter,
        'n_train'      : len(es_train_df),
        'n_val'        : len(es_val_df),
        'n_countries'  : len(available),
    }

    del loco_model, es_train_df, es_val_df, group_df
    del val_proba, kz_proba_raw
    gc.collect()

# =============================================================================
# ENSEMBLE & ATTACH TO KZ DATAFRAME
# =============================================================================
print(f"\n{'='*70}")
print("ENSEMBLE RESULTS\n")

for g, res in loco_results.items():
    kz_df[f'prob_{g}_raw'] = res['kz_proba_raw']
    kz_df[f'prob_{g}_cal'] = res['kz_proba_cal']

# Calibrated ensemble (recommended — use this for downstream analysis)
cal_cols = [f'prob_{g}_cal' for g in loco_results]
kz_df['prob_loco_ensemble'] = kz_df[cal_cols].mean(axis=1)
kz_df['pred_loco_ensemble'] = (kz_df['prob_loco_ensemble'] >= 0.5).astype(int)

# Raw ensemble (for comparison)
raw_cols = [f'prob_{g}_raw' for g in loco_results]
kz_df['prob_loco_ensemble_raw'] = kz_df[raw_cols].mean(axis=1)

# =============================================================================
# SUMMARY TABLE
# =============================================================================
summary_rows = []
for g, res in loco_results.items():
    summary_rows.append({
        'group'            : g,
        'n_countries'      : res['n_countries'],
        'n_train'          : res['n_train'],
        'n_val'            : res['n_val'],
        'best_iter'        : res['best_iter'],
        'val_AUC'          : round(res['val_auc'],       4),
        'val_AUPRC'        : round(res['val_auprc'],     4),
        'val_Brier_raw'    : round(res['val_brier_raw'], 4),
        'val_Brier_cal'    : round(res['val_brier_cal'], 4),
        'kz_mean_prob_raw' : round(res['kz_proba_raw'].mean(), 4),
        'kz_mean_prob_cal' : round(res['kz_proba_cal'].mean(), 4),
        'kz_pct_resistant' : round((res['kz_proba_cal'] >= 0.5).mean() * 100, 1),
    })

summary_df = pd.DataFrame(summary_rows)

print(summary_df.to_string(index=False))

# =============================================================================
# SAVE
# =============================================================================
kz_df.to_csv(f'{OUT_DIR}/kz_loco_fixed.csv',          index=False)
summary_df.to_csv(f'{OUT_DIR}/loco_fixed_summary.csv', index=False)

print(f"\nFIXED LOCO Complete")
print(f"   kz_loco_fixed.csv          — KZ predictions per organism/antibiotic/year")
print(f"   loco_fixed_summary.csv     — group-level val metrics")
for g in loco_results:
    print(f"   loco_{g}.cbm  — saved model")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

kz_df = pd.read_csv('/kaggle/working/kz_loco_fixed.csv')

# ── Yearly × Organism (averaged across antibiotics) ───────────────────────────
yearly_org = (
    kz_df.groupby(['year', 'organism'])['prob_loco_ensemble']
    .mean()
    .reset_index()
    .pivot(index='organism', columns='year', values='prob_loco_ensemble')
)

# Clean organism names (short for paper)
ORG_LABELS = {
    'Escherichia coli':        'E. coli',
    'Klebsiella pneumoniae':   'K. pneumoniae',
    'Staphylococcus aureus':   'S. aureus',
    'Pseudomonas aeruginosa':  'P. aeruginosa',
    'Acinetobacter baumannii': 'A. baumannii',
    'Enterococcus faecium':    'E. faecium',
}
yearly_org.index = [ORG_LABELS.get(i, i) for i in yearly_org.index]

# ── FIGURE ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))

sns.heatmap(
    yearly_org,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn_r',
    vmin=0, vmax=1,
    linewidths=0.6,
    linecolor='white',
    annot_kws={'size': 10, 'weight': 'bold'},
    cbar_kws={
        'label': 'Predicted Resistance Probability',
        'shrink': 0.85,
        'orientation': 'vertical'
    },
    ax=ax
)

#ax.set_title(
##    'Predicted Antimicrobial Resistance Probability in Kazakhstan (2017–2023)\n'
#   'Zero-shot LOCO CatBoost ensemble — calibrated probabilities',
#    fontsize=12, fontweight='bold', pad=12
#)
ax.set_xlabel('Year', fontsize=11, labelpad=8)
ax.set_ylabel('Pathogen', fontsize=11, labelpad=8)
ax.tick_params(axis='x', labelsize=10, rotation=0)
ax.tick_params(axis='y', labelsize=10, rotation=0)

# Italic y-axis labels (species names convention)
ax.set_yticklabels(
    [f'$\it{{{label.replace(" ", "\ ")}}}$' for label in yearly_org.index],
    fontsize=10
)

plt.tight_layout()

# Save as high-res for paper (300 dpi, PDF also provided)
plt.savefig('/kaggle/working/kz_heatmap_paper.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/kaggle/working/kz_heatmap_paper.pdf',
            bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved: kz_heatmap_paper.png + kz_heatmap_paper.pdf")

# ── ALSO SAVE THE UNDERLYING TABLE ────────────────────────────────────────────
yearly_org.round(3).to_csv('/kaggle/working/kz_yearly_organism_resistance.csv')
print("✅ Saved: kz_yearly_organism_resistance.csv")

In [ ]:
import gc
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt
import os

print("=== LOCO Validation for Kazakhstan (CatBoost + AutoGluon) ===\n")

# ----------------------------- CONFIG -----------------------------
ES_VAL_YEARS = [2019, 2020]
MODEL_DIR    = '/kaggle/input/datasets/uaisamangeldi/importmodels'

post_soviet      = ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia']
structural_peers = ['Qatar', 'Turkey', 'Romania', 'Bulgaria', 'Hungary', 'Poland']

# ----------------------------- LOAD DATA -----------------------------
full_df  = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv', low_memory=False)
kz_input = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/kzzzzz/kz_inference_filtered.csv')

print(f"Full dataset : {len(full_df):,} rows")
print(f"KZ macro rows: {len(kz_input):,} rows")

# ----------------------------- EXPAND KZ WITH ORGANISMS -----------------------------
TARGET_ORGANISMS = [
    'Escherichia coli', 'Klebsiella pneumoniae', 'Staphylococcus aureus',
    'Pseudomonas aeruginosa', 'Acinetobacter baumannii', 'Enterococcus faecium'
]

ABX_CLASS_MAP = {
    'meropenem':     'Carbapenem',
    'ciprofloxacin': 'Fluoroquinolone',
    'ceftriaxone':   'Cephalosporin',
    'amikacin':      'Aminoglycoside',
    'ampicillin':    'Penicillin',
}

prediction_rows = []
for _, row in kz_input.iterrows():
    for org in TARGET_ORGANISMS:
        new_row = row.to_dict()
        new_row['organism']    = org
        new_row['country']     = 'Kazakhstan'
        new_row['data_source'] = 'KZ_DID'
        if 'antibiotic_class' not in new_row or pd.isna(new_row.get('antibiotic_class')):
            new_row['antibiotic_class'] = ABX_CLASS_MAP.get(
                str(new_row.get('antibiotic', '')).lower(), 'unknown'
            )
        prediction_rows.append(new_row)

kz_df = pd.DataFrame(prediction_rows)
print(f"KZ rows after organism expansion: {len(kz_df):,}\n")

# ----------------------------- LOAD PRE-TRAINED MODELS -----------------------------
print("Loading pre-trained models...")

cbm_model = CatBoostClassifier()
cbm_model.load_model(os.path.join(MODEL_DIR, 'all_cbm_files', 'catboost_model.cbm'))
print("CatBoost loaded")

from autogluon.tabular import TabularPredictor
ag_model = TabularPredictor.load(os.path.join(MODEL_DIR, 'autogluon_model_lean'))
print("AutoGluon loaded")

# ----------------------------- HELPERS -----------------------------
def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    for col in X.select_dtypes(include='float64').columns:
        X[col] = X[col].astype('float32')
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

def make_ag_df(df):
    """AutoGluon expects a plain DataFrame with the same columns used at training."""
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    return X

# ----------------------------- LOCO GROUPS -----------------------------
all_except_kz = [c for c in full_df['country'].unique() if c != 'Kazakhstan']

loco_groups = {
    'post_soviet':      post_soviet,
    'structural_peers': structural_peers,
    'all_except_kz':    all_except_kz,
}

# Build KZ inputs once
kz_pool  = make_pool(kz_df, has_labels=False)
kz_ag_df = make_ag_df(kz_df)

# ----------------------------- LOCO LOOP -----------------------------
loco_results = {}

for group_name, countries in loco_groups.items():
    print(f"\n{'='*65}")
    print(f"[{group_name.upper()}] — {len(countries)} countries")

    available = [c for c in countries if c in full_df['country'].unique()]
    missing   = set(countries) - set(available)
    if missing:
        print(f"  ⚠️  Not in dataset: {missing}")
    if not available:
        print("  Skipped — no data")
        continue

    group_df  = full_df[full_df['country'].isin(available)]
    es_val_df = group_df[group_df['year'].isin(ES_VAL_YEARS)].copy()

    if len(es_val_df) < 200:
        print(f"  Skipped — val set too small ({len(es_val_df)} rows)")
        continue

    print(f"  Val rows: {len(es_val_df):,}  |  KZ rows: {len(kz_df):,}")
    y_val = es_val_df['resistance'].values

    # ── CatBoost ──────────────────────────────────────────────────
    val_proba_cbm = cbm_model.predict_proba(make_pool(es_val_df))[:, 1]
    kz_proba_cbm  = cbm_model.predict_proba(kz_pool)[:, 1]

    # ── AutoGluon ─────────────────────────────────────────────────
    val_proba_ag = ag_model.predict_proba(make_ag_df(es_val_df))[1].values
    kz_proba_ag  = ag_model.predict_proba(kz_ag_df)[1].values

    # ── Ensemble ──────────────────────────────────────────────────
    val_proba_ens = (val_proba_cbm + val_proba_ag) / 2
    kz_proba_ens  = (kz_proba_cbm  + kz_proba_ag)  / 2

    # ── Metrics ───────────────────────────────────────────────────
    def metrics(y, p):
        return {
            'auc':   roc_auc_score(y, p) if y.std() > 0 else np.nan,
            'auprc': average_precision_score(y, p) if y.std() > 0 else np.nan,
            'brier': brier_score_loss(y, p),
        }

    m_cbm = metrics(y_val, val_proba_cbm)
    m_ag  = metrics(y_val, val_proba_ag)
    m_ens = metrics(y_val, val_proba_ens)

    print(f"  CatBoost  AUC={m_cbm['auc']:.4f} AUPRC={m_cbm['auprc']:.4f} Brier={m_cbm['brier']:.4f}")
    print(f"  AutoGluon AUC={m_ag['auc']:.4f} AUPRC={m_ag['auprc']:.4f} Brier={m_ag['brier']:.4f}")
    print(f"  Ensemble  AUC={m_ens['auc']:.4f} AUPRC={m_ens['auprc']:.4f} Brier={m_ens['brier']:.4f}")
    print(f"  KZ mean prob (ensemble): {kz_proba_ens.mean():.4f} | "
          f"Resistant: {(kz_proba_ens >= 0.5).mean():.1%}")

    loco_results[group_name] = {
        'kz_proba_cbm': kz_proba_cbm,
        'kz_proba_ag':  kz_proba_ag,
        'kz_proba_ens': kz_proba_ens,
        'kz_pred_ens':  (kz_proba_ens >= 0.5).astype(int),
        'val_metrics':  {'catboost': m_cbm, 'autogluon': m_ag, 'ensemble': m_ens},
        'n_val':        len(es_val_df),
        'n_countries':  len(available),
    }

    del group_df, es_val_df
    gc.collect()

# ----------------------------- ENSEMBLE ACROSS GROUPS ----------------------------
print(f"\n{'='*65}")
print("FINAL ENSEMBLE (mean across LOCO groups)\n")

for g, res in loco_results.items():
    kz_df[f'prob_{g}'] = res['kz_proba_ens']

prob_cols = [f'prob_{g}' for g in loco_results]
kz_df['prob_ensemble'] = kz_df[prob_cols].mean(axis=1)
kz_df['pred_ensemble'] = (kz_df['prob_ensemble'] >= 0.5).astype(int)

print("KZ Prediction Distribution (final ensemble):")
print(kz_df['pred_ensemble'].value_counts(normalize=True).round(3))

# ----------------------------- SUMMARY TABLE ------------------------------------
rows = []
for g, res in loco_results.items():
    for model_name, m in res['val_metrics'].items():
        rows.append({
            'group':            g,
            'model':            model_name,
            'n_countries':      res['n_countries'],
            'n_val':            res['n_val'],
            'val_AUC':          round(m['auc'],   4),
            'val_AUPRC':        round(m['auprc'], 4),
            'val_Brier':        round(m['brier'], 4),
            'kz_mean_prob':     round(res['kz_proba_ens'].mean(), 4),
            'kz_pct_resistant': round(res['kz_pred_ens'].mean() * 100, 1),
        })

summary_df = pd.DataFrame(rows)
print("\n", summary_df.to_string(index=False))

# ----------------------------- SAVE ----------------------------------------------
# ----------------------------- ENSEMBLE ACROSS GROUPS ----------------------------
for g, res in loco_results.items():
    kz_df[f'prob_cbm_{g}'] = res['kz_proba_cbm']
    kz_df[f'prob_ag_{g}']  = res['kz_proba_ag']
    kz_df[f'prob_ens_{g}'] = res['kz_proba_ens']

# Aggregate across groups
cbm_cols = [f'prob_cbm_{g}' for g in loco_results]
ag_cols  = [f'prob_ag_{g}'  for g in loco_results]
ens_cols = [f'prob_ens_{g}' for g in loco_results]

kz_df['prob_catboost']  = kz_df[cbm_cols].mean(axis=1)
kz_df['prob_autogluon'] = kz_df[ag_cols].mean(axis=1)
kz_df['prob_ensemble']  = kz_df[ens_cols].mean(axis=1)

kz_df['pred_catboost']  = (kz_df['prob_catboost']  >= 0.5).astype(int)
kz_df['pred_autogluon'] = (kz_df['prob_autogluon'] >= 0.5).astype(int)
kz_df['pred_ensemble']  = (kz_df['prob_ensemble']  >= 0.5).astype(int)

kz_df.to_csv('/kaggle/working/kz_final_with_loco_preds.csv', index=False)

# ----------------------------- HEATMAPS SIDE BY SIDE ----------------------------
import seaborn as sns
import matplotlib.pyplot as plt

def make_pivot(prob_col):
    return (
        kz_df.groupby(['organism', 'antibiotic'])[prob_col]
        .mean()
        .unstack(fill_value=0)
    )

pivot_cbm = make_pivot('prob_catboost')
pivot_ag  = make_pivot('prob_autogluon')
pivot_ens = make_pivot('prob_ensemble')

n_abx = len(pivot_cbm.columns)
n_org = len(pivot_cbm.index)
cell_w, cell_h = 1.2, 1.0

fig, axes = plt.subplots(
    1, 3,
    figsize=(max(16, n_abx * cell_w) * 3, max(8, n_org * cell_h))
)

for ax, pivot, title in zip(
    axes,
    [pivot_cbm, pivot_ag, pivot_ens],
    ['CatBoost Only', 'AutoGluon Only', 'Ensemble (CBM + AG)']
):
    sns.heatmap(
        pivot,
        annot=True, fmt='.2f',
        cmap='RdYlGn_r',
        vmin=0, vmax=1,
        linewidths=0.5, linecolor='white',
        annot_kws={'size': 9},
        ax=ax,
        cbar_kws={'shrink': 0.6}
    )
    ax.set_title(f'Kazakhstan — {title}\nMean Predicted Resistance Probability',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Antibiotic', fontsize=11)
    ax.set_ylabel('Organism', fontsize=11)
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    ax.tick_params(axis='y', labelsize=9, rotation=0)

plt.suptitle('Resistance Probability Comparison: CatBoost vs AutoGluon vs Ensemble',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/kz_heatmap_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# ----------------------------- DISAGREEMENT TABLE --------------------------------
# Where do CatBoost and AutoGluon disagree most?
kz_df['prob_disagreement'] = (kz_df['prob_catboost'] - kz_df['prob_autogluon']).abs()

disagreement = (
    kz_df.groupby(['organism', 'antibiotic'])['prob_disagreement']
    .mean()
    .reset_index()
    .sort_values('prob_disagreement', ascending=False)
)

print("\nTop 10 organism-antibiotic pairs with highest model disagreement:")
print(disagreement.head(10).to_string(index=False))
disagreement.to_csv('/kaggle/working/kz_model_disagreement.csv', index=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def make_pivot(prob_col):
    return (
        kz_df.groupby(['organism', 'antibiotic'])[prob_col]
        .mean()
        .unstack(fill_value=0)
    )

pivot_cbm = make_pivot('prob_catboost')
pivot_ag  = make_pivot('prob_autogluon')
pivot_ens = make_pivot('prob_ensemble')

n_abx = len(pivot_cbm.columns)
n_org = len(pivot_cbm.index)

cell_w = 1.4
cell_h = 1.1
fig_w  = max(20, n_abx * cell_w)
fig_h  = max(10, n_org * cell_h)

for pivot, title, fname in [
    (pivot_cbm, 'CatBoost Only',          'kz_heatmap_catboost.png'),
    (pivot_ag,  'AutoGluon Only',          'kz_heatmap_autogluon.png'),
    (pivot_ens, 'Ensemble (CBM + AG)',     'kz_heatmap_ensemble.png'),
]:
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sns.heatmap(
        pivot,
        annot=True,
        fmt='.2f',
        cmap='RdYlGn_r',
        vmin=0, vmax=1,
        linewidths=0.5,
        linecolor='white',
        annot_kws={'size': 10},
        ax=ax,
        cbar_kws={'shrink': 0.6, 'label': 'Resistance Probability'}
    )

    ax.set_title(
        f'Kazakhstan — {title}\nMean Predicted Resistance Probability (Organism × Antibiotic)',
        fontsize=14, fontweight='bold', pad=15
    )
    ax.set_xlabel('Antibiotic', fontsize=12, labelpad=10)
    ax.set_ylabel('Organism',   fontsize=12, labelpad=10)
    ax.tick_params(axis='x', labelsize=10, rotation=45)
    ax.tick_params(axis='y', labelsize=10, rotation=0)

    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{fname}', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved: {fname}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Kazakhstan predictions
kz_df = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/kzzzzz/kz_loco_fixed (1).csv')

PROB_COL = 'prob_loco_ensemble'          # calibrated ensemble probability



risk_table = (
    kz_df.groupby(['year', 'antibiotic', 'organism'])[PROB_COL]
    .mean()
    .reset_index()
    .rename(columns={PROB_COL: 'predicted_risk'})
    .sort_values(['year', 'predicted_risk'], ascending=[True, False])
)

print(risk_table.head(20).to_string(index=False))
risk_table.to_csv('/kaggle/working/kz_risk_year_abx_org.csv', index=False)


print("\nTOP 5 HIGHEST RISK COMBINATIONS PER YEAR:\n")
for year in sorted(risk_table['year'].unique()):
    top5 = risk_table[risk_table['year'] == year].head(5)
    print(f"  {year}:")
    print(top5[['antibiotic', 'organism', 'predicted_risk']]
          .round(4).to_string(index=False))
    print()


pivot = (
    risk_table.groupby(['organism', 'antibiotic'])['predicted_risk']
    .mean()
    .unstack(fill_value=0)
)

n_abx = len(pivot.columns)
n_org = len(pivot.index)

fig, ax = plt.subplots(figsize=(max(16, n_abx * 1.2), max(8, n_org * 1.0)))

sns.heatmap(
    pivot,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn_r',          # green = low risk, red = high risk
    vmin=0, vmax=1,
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 9},
    ax=ax
)

#ax.set_title('Kazakhstan — Mean Calibrated Predicted Resistance Probability\n(Organism × Antibiotic)',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Antibiotic', fontsize=12)
ax.set_ylabel('Organism', fontsize=12)
ax.tick_params(axis='x', labelsize=9, rotation=45)
ax.tick_params(axis='y', labelsize=10, rotation=0)

plt.tight_layout()
plt.savefig('/kaggle/working/kz_resistance_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

# ── 4. TREND OVER TIME — top organism-antibiotic pairs ────────────────────────
# Find top 5 highest-risk pairs overall (by mean calibrated risk)
top_pairs = (
    risk_table.groupby(['antibiotic', 'organism'])['predicted_risk']
    .mean()
    .nlargest(5)
    .reset_index()
)
top_pairs['label'] = top_pairs['organism'].str.split().str[-1] + ' / ' + top_pairs['antibiotic']

plt.figure(figsize=(12, 5))
for _, pair in top_pairs.iterrows():
    subset = risk_table[
        (risk_table['antibiotic'] == pair['antibiotic']) &
        (risk_table['organism']   == pair['organism'])
    ]
    plt.plot(subset['year'], subset['predicted_risk'],
             marker='o', label=pair['label'])

plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Decision threshold (0.5)')
#plt.title('Kazakhstan — Calibrated Resistance Probability Trend (Top 5 Pairs)', fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Mean Calibrated Predicted Resistance Probability')
plt.legend(fontsize=9, bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('/kaggle/working/kz_resistance_trend.png', dpi=150)
plt.show()

# ── 5. BINARY RESISTANCE RATE TABLE (using binary predictions) ─────────────────
# Instead of multiplying probability by 100, use the existing binary predictions
# (pred_loco_ensemble) to compute actual percentage classified as resistant.
binary_table = (
    kz_df.groupby(['year', 'antibiotic', 'organism'])['pred_loco_ensemble']
    .mean()                         
    .mul(100).round(1)
    .reset_index()
    .rename(columns={'pred_loco_ensemble': 'pct_predicted_resistant'})
    .sort_values(['year', 'pct_predicted_resistant'], ascending=[True, False])
)
binary_table.to_csv('/kaggle/working/kz_binary_resistance_rate.csv', index=False)
print("\nBinary resistance rate (% of organism-antibiotic combos predicted resistant):")
print(binary_table.head(15).to_string(index=False))

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, recall_score, precision_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# LOAD MODEL + DATA
# =============================================================================
MODEL_DIR = '/kaggle/input/datasets/uaisamangeldi/importmodels'

model_final = CatBoostClassifier()
model_final.load_model(os.path.join(MODEL_DIR, 'all_cbm_files', 'catboost_model.cbm'))
FEATURES = CAT_FEATURES + NUM_FEATURES

full_df = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv', low_memory=False)
full_df = full_df.loc[:, ~full_df.columns.duplicated()]

EXCLUDE_COUNTRIES = ['Kazakhstan', 'Russia', 'Belarus', 'Ukraine']
source_df = full_df[~full_df['country'].isin(EXCLUDE_COUNTRIES)].copy()
del full_df; gc.collect()

TEMPORAL_CUTOFF = 2020
train_df = source_df[source_df['year'] <= TEMPORAL_CUTOFF].copy()
test_df  = source_df[source_df['year'] > TEMPORAL_CUTOFF].copy()

print(f"Train rows : {len(train_df):,}")
print(f"Test rows  : {len(test_df):,}")

def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_bin = (y_prob >= threshold).astype(int)
    bs    = brier_score_loss(y_true, y_prob)
    prev  = y_true.mean()
    bss   = 1.0 - bs / (prev * (1 - prev)) if prev > 0 else np.nan
    return {
        'AUC':       roc_auc_score(y_true, y_prob),
        'AUPRC':     average_precision_score(y_true, y_prob),
        'F1':        f1_score(y_true, y_bin, zero_division=0),
        'Precision': precision_score(y_true, y_bin, zero_division=0),
        'Recall':    recall_score(y_true, y_bin, zero_division=0),
        'Brier':     bs,
        'BSS':       bss,
    }

# scale_pos_weight computed on train only (fix: was previously computed on full source_df)
SCALE_POS_WEIGHT = (train_df['resistance'] == 0).sum() / (train_df['resistance'] == 1).sum()
FINAL_ROUNDS     = model_final.tree_count_

BASE_PARAMS = {
    'learning_rate':         0.03,
    'depth':                 7,
    'l2_leaf_reg':           3,
    'eval_metric':           'AUC',
    'random_seed':           42,
    'verbose':               0,
    'early_stopping_rounds': None,
    'iterations':            FINAL_ROUNDS,
    'task_type':             'GPU',
    'scale_pos_weight':      SCALE_POS_WEIGHT,
}

# =============================================================================
# 1. TEST SET EVALUATION
# =============================================================================
print("\n" + "="*70)
print("1. TEST SET EVALUATION")
print("="*70)

y_te   = test_df['resistance'].values
y_prob = model_final.predict_proba(make_pool(test_df))[:, 1]
metrics = compute_metrics(y_te, y_prob)

for k, v in metrics.items():
    print(f"  {k:<12}: {v:.4f}")

y_pred = (y_prob >= 0.5).astype(int)
cm     = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix with labeled counts
ax = axes[0]
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred S', 'Pred R']); ax.set_yticklabels(['True S', 'True R'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
labels = [
    [f'TN\n{tn:,}', f'FP\n{fp:,}'],
    [f'FN\n{fn:,}', f'TP\n{tp:,}'],
]
thresh = cm.max() / 2.0
for i in range(2):
    for j in range(2):
        ax.text(j, i, labels[i][j], ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black', fontsize=12)
plt.colorbar(im, ax=ax)

fpr, tpr, _ = roc_curve(y_te, y_prob)
axes[1].plot(fpr, tpr, color='steelblue', label=f'AUC = {metrics["AUC"]:.4f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_te, y_prob)
axes[2].plot(rec, prec, color='darkorange', label=f'AUPRC = {metrics["AUPRC"]:.4f}')
axes[2].axhline(y_te.mean(), color='gray', linestyle='--', label='Baseline')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_evaluation_plots.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_evaluation_plots.png")

# =============================================================================
# 2. FEATURE IMPORTANCE
# =============================================================================
print("\n" + "="*70)
print("2. FEATURE IMPORTANCE")
print("="*70)

fi = pd.DataFrame({
    'feature':    FEATURES,
    'importance': model_final.get_feature_importance()
}).sort_values('importance', ascending=False)

fi.to_csv(f'{OUTPUT_DIR}/cbm_feature_importance.csv', index=False)
print(fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, max(6, len(FEATURES) * 0.4)))
colors = ['#d73027' if i < 3 else '#4575b4' for i in range(len(fi))]
ax.barh(fi['feature'][::-1], fi['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Importance Score', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_feature_importance.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_feature_importance.png")

# =============================================================================
# 3. SHAP ANALYSIS
# =============================================================================
print("\n" + "="*70)
print("3. SHAP ANALYSIS")
print("="*70)

shap_sample = test_df.sample(min(5000, len(test_df)), random_state=42)
explainer   = shap.TreeExplainer(model_final)
shap_values = explainer.shap_values(make_pool(shap_sample, has_labels=False))

shap.summary_plot(shap_values, shap_sample[FEATURES], max_display=15, show=False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_shap_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_shap_summary.png")

shap.summary_plot(shap_values, shap_sample[FEATURES], plot_type='bar', max_display=15, show=False)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_shap_bar.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_shap_bar.png")

top3_numeric = fi[fi['feature'].isin(NUM_FEATURES)]['feature'].head(3).tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top3_numeric):
    feat_idx = FEATURES.index(feat)
    shap.dependence_plot(
        feat_idx, shap_values, shap_sample[FEATURES],
        interaction_index=None, ax=ax, show=False
    )
    ax.set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_shap_dependence.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_shap_dependence.png")

# =============================================================================
# 4. 5-FOLD TIME-AWARE CROSS VALIDATION
# =============================================================================
print("\n" + "="*70)
print("4. 5-FOLD TIME-AWARE CROSS VALIDATION")
print("="*70)

CV_FOLDS = [
    (2015, 2016), (2016, 2017), (2017, 2018),
    (2018, 2019), (2019, 2020)
]
cv_results = []

for train_end, val_year in CV_FOLDS:
    fold_train = source_df[source_df['year'] <= train_end].copy()
    fold_val   = source_df[source_df['year'] == val_year].copy()

    if len(fold_val) < 100:
        print(f"  Fold {train_end}→{val_year}: skipped (val too small)")
        continue

    # scale_pos_weight computed per fold on fold_train only
    fold_spw = (fold_train['resistance'] == 0).sum() / (fold_train['resistance'] == 1).sum()
    fold_iterations = max(100, int(FINAL_ROUNDS * (len(fold_train) / len(train_df))))

    fold_params = {**BASE_PARAMS, 'scale_pos_weight': fold_spw, 'iterations': fold_iterations}
    fold_model  = CatBoostClassifier(**fold_params)
    fold_model.fit(make_pool(fold_train))

    y_prob_fold  = fold_model.predict_proba(make_pool(fold_val))[:, 1]
    fold_metrics = compute_metrics(fold_val['resistance'].values, y_prob_fold)
    fold_metrics.update({'train_end': train_end, 'val_year': val_year,
                         'n_train': len(fold_train), 'n_val': len(fold_val)})
    cv_results.append(fold_metrics)
    print(f"  Fold {train_end}→{val_year}  AUC={fold_metrics['AUC']:.4f}  "
          f"AUPRC={fold_metrics['AUPRC']:.4f}  F1={fold_metrics['F1']:.4f}")

    del fold_model, fold_train, fold_val; gc.collect()

cv_df = pd.DataFrame(cv_results)
print(f"\n  Mean AUC  : {cv_df['AUC'].mean():.4f} ± {cv_df['AUC'].std():.4f}")
print(f"  Mean AUPRC: {cv_df['AUPRC'].mean():.4f} ± {cv_df['AUPRC'].std():.4f}")
print(f"  Mean F1   : {cv_df['F1'].mean():.4f} ± {cv_df['F1'].std():.4f}")
cv_df.to_csv(f'{OUTPUT_DIR}/cbm_cv_results.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(cv_df))
for metric, color in [('AUC','steelblue'), ('AUPRC','darkorange'), ('F1','green')]:
    ax.plot(x, cv_df[metric], marker='o', label=metric, color=color)
ax.set_xticks(x)
ax.set_xticklabels([f"{int(r['train_end'])}→{int(r['val_year'])}"
                    for _, r in cv_df.iterrows()], rotation=20)
ax.set_ylim(0, 1)
ax.set_ylabel('Score'); ax.set_xlabel('Fold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_cv_plot.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_cv_results.csv + cbm_cv_plot.png")

# =============================================================================
# 5. ABLATION ANALYSIS
# =============================================================================
print("\n" + "="*70)
print("5. ABLATION ANALYSIS")
print("="*70)

ABLATION_GROUPS = {
    'All features (baseline)':            FEATURES,
    'Drop antibiotic_class':              [f for f in FEATURES if f != 'antibiotic_class'],
    'Drop macro (gdp/health)':            [f for f in FEATURES if f not in
                                           ['log_gdp_per_capita','health_expenditure_pct_gdp']],
    'Drop consumption':                   [f for f in FEATURES if f not in
                                           ['log_consumption_ddd','log_consumption_did_per_class']],
    'Drop infra (sanitation/water/beds)': [f for f in FEATURES if f not in
                                           ['sanitation_access_pct','water_access_pct',
                                            'hospital_beds_per_1000']],
    'Drop climate (temp/aware)':          [f for f in FEATURES if f not in
                                           ['mean_temp_celsius','aware_access_pct']],
    'Cats only':                          CAT_FEATURES,
    'Nums only':                          NUM_FEATURES,
}

ablation_results = []

for group_label, feat_subset in ABLATION_GROUPS.items():
    cat_sub = [c for c in CAT_FEATURES if c in feat_subset]

    def make_pool_ablation(df, has_labels=True, feats=feat_subset, cats=cat_sub):
        X = df[feats].copy()
        for col in cats:
            X[col] = X[col].fillna('unknown').astype(str)
        if has_labels:
            return Pool(data=X, label=df['resistance'].values, cat_features=cats)
        return Pool(data=X, cat_features=cats)

    # scale_pos_weight from train_df only
    abl_spw    = (train_df['resistance'] == 0).sum() / (train_df['resistance'] == 1).sum()
    abl_params = {**BASE_PARAMS, 'scale_pos_weight': abl_spw}
    abl_model  = CatBoostClassifier(**abl_params)
    abl_model.fit(make_pool_ablation(train_df))

    y_prob_abl = abl_model.predict_proba(make_pool_ablation(test_df, has_labels=False))[:, 1]
    abl_m = compute_metrics(y_te, y_prob_abl)
    abl_m['group']   = group_label
    abl_m['n_feats'] = len(feat_subset)
    ablation_results.append(abl_m)

    print(f"  {group_label:<42} AUC={abl_m['AUC']:.4f}  "
          f"AUPRC={abl_m['AUPRC']:.4f}  F1={abl_m['F1']:.4f}")

    del abl_model; gc.collect()

abl_df = pd.DataFrame(ablation_results)
abl_df.to_csv(f'{OUTPUT_DIR}/cbm_ablation_results.csv', index=False)

baseline_auc   = abl_df.loc[abl_df['group'] == 'All features (baseline)', 'AUC'].values[0]
abl_df['AUC_drop'] = baseline_auc - abl_df['AUC']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2166ac' if g == 'All features (baseline)' else
          '#d73027' if v > 0.01 else '#fee090'
          for g, v in zip(abl_df['group'], abl_df['AUC_drop'])]

axes[0].barh(abl_df['group'][::-1], abl_df['AUC'][::-1], color=colors[::-1])
axes[0].axvline(baseline_auc, color='black', linestyle='--', alpha=0.5, label='Baseline')
axes[0].set_xlabel('AUC')
axes[0].legend()

axes[1].barh(abl_df['group'][::-1], abl_df['AUC_drop'][::-1],
             color=['#d73027' if v > 0 else '#1a9850' for v in abl_df['AUC_drop'][::-1]])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('AUC Drop vs Baseline')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cbm_ablation.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: cbm_ablation_results.csv + cbm_ablation.png")

print("\n" + "="*70)
print("ALL ANALYSES COMPLETE")
print("="*70)